# LG-CXR FRCNN Standalone Colab Bootstrap

Use this notebook when you only copied/pasted the notebook into Colab and do not have the `amia-lgcxr-frcnn/` folder uploaded. It recreates the project under `/content/amia-lgcxr-frcnn`, then runs the same Faster R-CNN workflow.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Create Project Files

This cell writes the project source code, scripts, config, and docs into `/content/amia-lgcxr-frcnn`. Re-run it if the Colab runtime resets.

In [ ]:
from pathlib import Path
import os

PROJECT_DIR = Path('/content/amia-lgcxr-frcnn')
PROJECT_FILES = {
  "requirements.txt": "torch\ntorchvision\npandas\nnumpy\nPillow\nPyYAML\ntqdm\nmatplotlib\ntimm\n",
  "README.md": "# LG-CXR FRCNN Baseline: Local-Global Chest X-ray Detection with Faster R-CNN\n\nThis project builds a reproducible baseline for the AMIA Public Challenge 2026 using a license-friendlier PyTorch stack. V1 uses `torchvision` Faster R-CNN ResNet50-FPN as the local scanner/detector, writes Kaggle-valid `submission.csv` files, and keeps optional ResNet18 global and crop classifiers as future extensions.\n\nThe competition evaluates standard PASCAL VOC 2010 mean Average Precision at IoU > 0.4. This repository therefore prioritizes clean box prediction, correct class mapping, and exact submission formatting before model complexity.\n\n## Why This Project Exists\n\n- Provide a YOLO-free and Ultralytics-free baseline.\n- Give humans and agents a clean project that can be reproduced from the README.\n- Run preflight checks before spending GPU time.\n- Produce leaderboard-valid output even when no detections survive filtering.\n- Make the first working path Colab-friendly while preserving Kaggle-style paths.\n\n## Model Idea\n\nV1 is:\n\n```text\nFaster R-CNN ResNet50-FPN -> filtered detections -> submission.csv\n```\n\nPlanned optional extensions are:\n\n- ResNet18 global classifier: predicts image-level pathology probabilities and can reweight detector scores later.\n- ResNet18 crop classifier: verifies candidate detection crops later.\n- Fusion logic: V1 is pass-through; learned fusion is intentionally deferred.\n\nOptional classifier checkpoints may be absent. The code must continue with Faster R-CNN only.\n\n## Theory\n\nObject detection predicts both what is present and where it appears. Classification predicts what is present in the whole image, but not the box location. Chest X-ray pathology tasks often need detection because the leaderboard expects class IDs, confidence scores, and bounding boxes.\n\nFaster R-CNN is a two-stage detector. The first stage, the Region Proposal Network, proposes likely object regions. The second stage classifies those proposals and refines their bounding boxes. The FPN backbone helps the detector reason across scales, which is useful for medical findings that may be small, diffuse, or low contrast.\n\nFaster R-CNN is a good first medical detection baseline because it is widely available in `torchvision`, avoids YOLO/Ultralytics licensing concerns, is simple to checkpoint, and can be trained with ordinary PyTorch loops.\n\nGlobal image-level classification can help later because some findings may be visually subtle but globally likely. Crop-level verification can help later by rescoring candidate boxes after the detector proposes them. These are not required for V1.\n\nClass `14` is `\"No finding\"`, but it is not a physical object. Training the detector to localize class `14` would teach it a fake box target. Instead, class `14` rows are treated as negative/no-finding annotations, and class `14` appears only as the submission fallback.\n\nValidation should use IoU threshold `0.4` because the challenge uses VOC mAP at IoU > 0.4.\n\n## Competition Rules\n\nSubmission columns must be exactly:\n\n```text\nimage_id,PredictionString\n```\n\nEach `PredictionString` contains repeated groups:\n\n```text\nclass_id confidence xmin ymin xmax ymax\n```\n\nReal pathology classes are original competition IDs `0..13`. Class ID `14` is `\"No finding\"` fallback only.\n\nIf a test image has no detections, output exactly:\n\n```text\n14 1.0 0 0 1 1\n```\n\nNever output an empty `PredictionString`.\n\nTorchvision detector labels use `0` for background, so the mapping is:\n\n```text\ncompetition 0 -> detector 1\ncompetition 1 -> detector 2\n...\ncompetition 13 -> detector 14\n```\n\nDetector predictions are mapped back to competition IDs `0..13` before submission.\n\n## Environment\n\nKaggle defaults:\n\n```text\nDATA_ROOT=/kaggle/input/amia-public-challenge-2026\nWORK_DIR=/kaggle/working\n```\n\nExpected competition dataset layout:\n\n```text\nDATA_ROOT/\n  train/\n  test/\n  img_size.csv\n  sample_submission.csv\n  test.csv\n  train.csv\n```\n\nThe code now prefers these exact files and folders before falling back to heuristic discovery. Training image IDs come from the `train/` folder when it exists, so train images without valid object rows can still be used as negative/no-finding Faster R-CNN samples. Test prediction prefers IDs from `test.csv`, while final submission still uses `sample_submission.csv` as the source of truth for row order and required IDs.\n\n`img_size.csv` is used by preflight to infer image dimensions and report whether object boxes fit within the declared image width and height. The image resolver accepts common ID variants such as bare stems, filenames, and path-like strings.\n\nImportant coordinate-space rule:\n\n- `train.csv` bounding boxes are in original scan coordinate space.\n- `img_size.csv` stores original dimensions per image. In this dataset, `dim0` is original height and `dim1` is original width.\n- The PNG files in `train/` and `test/` may be resized, commonly around `1024x1024`.\n- Faster R-CNN training targets must be in the actual PNG tensor coordinate space.\n- Kaggle submission boxes must be written back in original coordinate space.\n\nThe code therefore performs two conversions:\n\n```text\ntraining:   original train.csv boxes -> PNG image boxes -> Faster R-CNN\ninference:  Faster R-CNN PNG-space boxes -> original-space boxes -> submission.csv\n```\n\nDo not divide coordinates by a constant like `1024`. Original dimensions vary per image.\n\nColab defaults used in the notebook:\n\n```text\nPROJECT_DIR=/content/drive/MyDrive/amia-lgcxr-frcnn\nDATA_ROOT=/content/drive/MyDrive/datasets/amia-public-challenge-2026\nWORK_DIR=/content/drive/MyDrive/amia-lgcxr-frcnn/outputs\n```\n\nGPU is recommended. Torch and torchvision are required. `timm` is optional for future classifier work. YOLO, Ultralytics, and RT-DETR are not used for V1.\n\nRuntime path overrides are supported through CLI flags or environment variables:\n\n```bash\npython scripts/00_preflight.py --config configs/baseline_frcnn.yaml --data-root \"$DATA_ROOT\" --work-dir \"$WORK_DIR\"\n```\n\nor:\n\n```bash\nexport LGCXR_DATA_ROOT=/path/to/dataset\nexport LGCXR_WORK_DIR=/path/to/outputs\n```\n\n## Colab Setup\n\nOpen:\n\n```text\nnotebooks/LG_CXR_FRCNN_Colab.ipynb\n```\n\nUse this notebook when the full `amia-lgcxr-frcnn/` project folder already exists in Google Drive or has been uploaded to Colab. It includes cells to mount Google Drive, locate the project, install requirements, check CUDA, configure dataset and output paths, run preflight, run smoke test, train, predict, and make a submission.\n\nIf you plan to copy and paste a notebook into Colab without uploading the project folder, use the standalone bootstrap notebook instead:\n\n```text\nnotebooks/LG_CXR_FRCNN_Colab_Standalone.ipynb\n```\n\nThe standalone notebook creates the project files under:\n\n```text\n/content/amia-lgcxr-frcnn\n```\n\nThen it installs dependencies and runs the same preflight, smoke test, training, inference, and submission commands. This is the safer notebook when you do not have the folder in Drive yet.\n\nPlace the dataset at:\n\n```text\n/content/drive/MyDrive/datasets/amia-public-challenge-2026\n```\n\nor adjust `DATA_ROOT` in the notebook. All important Colab outputs are saved to `WORK_DIR`, including checkpoints, predictions, debug images, training summaries, and `submission.csv`.\n\n## Step-by-Step Usage\n\nRun preflight first:\n\n```bash\npython scripts/00_preflight.py --config configs/baseline_frcnn.yaml\n```\n\nAudit image dimensions and box scale:\n\n```bash\npython scripts/05_audit_dimensions.py --config configs/baseline_frcnn.yaml\n```\n\nTrain the scanner:\n\n```bash\npython scripts/01_train_scanner.py --config configs/baseline_frcnn.yaml\n```\n\nPredict validation and test:\n\n```bash\npython scripts/02_predict_scanner.py --config configs/baseline_frcnn.yaml\n```\n\nCreate the final submission:\n\n```bash\npython scripts/03_make_submission.py --config configs/baseline_frcnn.yaml\n```\n\nSingle-command pipeline:\n\n```bash\npython scripts/04_full_pipeline.py --config configs/baseline_frcnn.yaml\n```\n\nSmoke test:\n\n```bash\npython scripts/04_full_pipeline.py --config configs/baseline_frcnn.yaml --smoke-test\n```\n\nSmoke mode uses small subsets, trains for at most one epoch, and creates a format-valid submission. It is for plumbing checks, not score.\n\n## Preflight\n\nPreflight must run before training. It checks:\n\n- Exact Kaggle layout files and folders when present.\n- Dataset root exists.\n- CSV files load.\n- Train CSV is identified.\n- Test CSV is identified.\n- `img_size.csv` is identified when present.\n- Sample submission is identified.\n- Train and test folders contain images.\n- Train CSV image IDs resolve inside `train/`.\n- Test CSV and sample submission image IDs resolve inside `test/`.\n- Required columns are inferred.\n- 50 train images open when available.\n- 20 test images open when available.\n- Bounding boxes are valid for object classes.\n- Bounding boxes are checked against `img_size.csv` when possible.\n- Class mapping is valid.\n- Class `14` is handled as no-finding, not as a detection object.\n- Sample submission format is understood.\n- A small debug image is saved.\n- Training does not start if critical checks fail.\n\nPreflight writes:\n\n```text\nWORK_DIR/lgcxr_preflight_status.json\nWORK_DIR/debug_preflight_annotation.png\n```\n\n## Dimension Audit\n\nBefore changing `scanner.image_size`, run:\n\n```bash\npython scripts/05_audit_dimensions.py --config configs/baseline_frcnn.yaml\n```\n\nThis writes:\n\n```text\nWORK_DIR/lgcxr_dimension_audit.json\n```\n\nThe audit studies:\n\n- Dimensions declared in `img_size.csv`.\n- Real PIL image dimensions from `train/` and `test/`.\n- Whether original image dimensions differ from PNG dimensions.\n- Whether `train.csv` IDs resolve to files in `train/`.\n- Whether `test.csv` IDs resolve to files in `test/`.\n- Box width, height, and area in original pixels.\n- Box width, height, and area as fractions of image size.\n- Box size after scaling from original coordinates to actual PNG coordinates.\n- Estimated box dimensions after the current torchvision resize.\n- Whether boxes look normalized instead of pixel-based.\n- Whether boxes fall outside declared image dimensions.\n\nCurrent V1 config:\n\n```yaml\nscanner:\n  image_size: 800\n  max_size: 1200\n```\n\nIn torchvision Faster R-CNN, this means the shortest side is resized to about `800`, unless that would make the longest side exceed `1200`. Aspect ratio is preserved. This is a conservative first baseline for GPU memory.\n\nHow to interpret the report:\n\n- If most images are around `1024x1024`, `800/1200` is usually reasonable for V1.\n- If most images are `2048x2048` or larger and many boxes become under about `8-12` pixels wide after resizing, try `1024/1536` or `1200/1800`.\n- If boxes look normalized, fix coordinate parsing before training.\n- If many boxes exceed image dimensions, inspect whether the CSV uses width/height boxes, pixel boxes, normalized boxes, or a different coordinate origin.\n- If `boxes_scaled_original_to_png` is nonzero, that is expected for this dataset and means the original-space fix is active.\n- If CUDA OOM occurs, keep `800/1200` or reduce batch size before raising dimensions.\n\n## Training\n\nThe primary detector is `torchvision` Faster R-CNN ResNet50-FPN. The builder prefers `fasterrcnn_resnet50_fpn_v2` when available and falls back to `fasterrcnn_resnet50_fpn`.\n\nWhen `DATA_ROOT/train/` exists, training and validation splits are made from the images in that folder, not only from the IDs that appear in `train.csv`. Images with no valid class `0..13` boxes are valid negative samples. Annotation rows with class `14` are ignored as objects.\n\nBecause `train.csv` boxes are original-space boxes, `CXRDetectionDataset` uses `img_size.csv` to scale every object box from original scan dimensions into the actual PNG size before passing targets to Faster R-CNN. This prevents clipping/degenerate labels when original coordinates are larger than the PNG dimensions.\n\nCheckpoint files:\n\n```text\nWORK_DIR/lgcxr_scanner_fasterrcnn_best.pth\nWORK_DIR/lgcxr_scanner_fasterrcnn_last.pth\n```\n\nIf `scanner.resume: true`, training resumes from the last checkpoint when it exists. AMP is used only when CUDA is available. The validation proxy uses VOC-style matching at IoU `0.4`. Outputs are saved under `WORK_DIR`.\n\nBatch size is intentionally small by default because Faster R-CNN is memory-heavy. If CUDA OOM occurs, reduce `scanner.batch_size` or `scanner.image_size`.\n\nDo not change `scanner.image_size` blindly. Run the dimension audit first, then choose the next experiment based on resized box-size statistics.\n\n## Inference\n\nInference loads the best scanner checkpoint, predicts validation and test images, applies confidence thresholding, applies class-wise NMS, maps internal labels back to original class IDs, and saves:\n\n```text\nWORK_DIR/lgcxr_fast_val_predictions.csv\nWORK_DIR/lgcxr_fast_test_predictions.csv\n```\n\nPrediction rows contain:\n\n```text\nimage_id,class_id,confidence,xmin,ymin,xmax,ymax\n```\n\nFor test inference, `test.csv` image IDs are preferred when available. Submission creation still matches predictions back to `sample_submission.csv` IDs using tolerant ID variants, so `abc123`, `abc123.png`, and path-like image IDs can still line up.\n\nWhen `img_size.csv` is available, scanner predictions are saved in original coordinate space after scaling from PNG/model image coordinates back to the original scan dimensions. This is the coordinate space expected by `submission.csv`.\n\n## Submission\n\nSubmission creation uses the sample submission as the source of truth for row order and test IDs. It asserts:\n\n- Columns are exactly `[\"image_id\", \"PredictionString\"]`.\n- Row count equals sample submission.\n- No missing values exist.\n- Every prediction group length is a multiple of 6.\n- Class IDs are in `0..14`.\n- Class `14` appears only alone as the exact no-finding fallback.\n\nFinal output:\n\n```text\nWORK_DIR/submission.csv\n```\n\nOn Kaggle this resolves to:\n\n```text\n/kaggle/working/submission.csv\n```\n\n## Reproducibility\n\nReproducibility controls:\n\n- Central config: `configs/baseline_frcnn.yaml`\n- Seed: `42`\n- Stable train/validation split from the config seed.\n- Checkpoint paths recorded in config.\n- Preflight status saved before training.\n- Training summary JSON saved after training.\n\nRerun from a clean environment by installing requirements, setting paths, running preflight, running smoke test, then running the full pipeline.\n\n## Troubleshooting\n\nDataset not found: check `data_root`, `LGCXR_DATA_ROOT`, or the notebook `DATA_ROOT`.\n\nSample submission not found: verify the dataset contains a CSV with `image_id` and `PredictionString`.\n\nRequired columns not inferred: inspect the train CSV and update `src/data/columns.py` with the dataset's column names.\n\nInvalid boxes: check annotation rows with negative coordinates, swapped corners, missing class IDs, or zero-area boxes.\n\nToo many unreadable images: verify image file extensions and dataset placement.\n\nCUDA not available: training still runs on CPU, but it will be slow.\n\nOOM: reduce `scanner.batch_size`, reduce `scanner.image_size`, or enable smoke mode.\n\nCheckpoint missing: run training before inference, or verify `scanner.best_ckpt` points to an existing file.\n\nSubmission format mismatch: run `scripts/03_make_submission.py`; it includes strict validation assertions.\n\n## Roadmap\n\n- Train longer.\n- Tune thresholds per class.\n- Add global ResNet18 classifier.\n- Add crop ResNet18 classifier.\n- Add learned fusion.\n- Add hard negative mining.\n- Add pseudo-labeling if competition rules allow.\n- Add WBF/ensembling.\n- Add more precise VOC mAP evaluation.\n\n## Agent / Codex / Claude Workflow Rules\n\nEvery agent working on this repository must:\n\n1. Read `README.md` before making changes.\n2. Read `AGENT_INSTRUCTIONS.md` before making changes.\n3. Check the current project structure before editing.\n4. Keep the README consistent with the code.\n5. Update README whenever behavior, commands, config, outputs, checkpoints, model choice, or submission logic changes.\n6. Never silently change the submission format.\n7. Never train class `14` as an object.\n8. Never output empty `PredictionString`.\n9. Always run preflight before training.\n10. Prefer small, testable changes.\n11. Record important changes in a README changelog section.\n12. Do not introduce YOLO/Ultralytics into this project.\n13. Do not over-engineer before a valid submission exists.\n14. Make every script runnable from the command line.\n15. Keep config centralized in YAML.\n16. Keep `notebooks/LG_CXR_FRCNN_Colab.ipynb` synchronized with script commands.\n17. Keep `notebooks/LG_CXR_FRCNN_Colab_Standalone.ipynb` synchronized when project files or script commands change.\n18. If a script command changes, update README, `AGENT_INSTRUCTIONS.md`, and both Colab notebooks.\n\n## Changelog\n\n- 2026-06-02: Initial V1 project scaffold with Colab-first workflow, Faster R-CNN baseline, preflight, training, inference, and submission scripts.\n- 2026-06-02: Added standalone Colab bootstrap notebook guidance for copy/paste use without uploading the project folder first.\n- 2026-06-02: Hardened dataset layout handling for explicit `train/`, `test/`, `train.csv`, `test.csv`, `img_size.csv`, and `sample_submission.csv`; training now includes folder-level negative images and preflight cross-checks CSV/image alignment.\n- 2026-06-02: Added `scripts/05_audit_dimensions.py` and explicit `scanner.max_size` so image resize decisions can be based on real image and box statistics.\n- 2026-06-02: Fixed original-coordinate bounding-box handling: train boxes are scaled from original scan space into PNG space for Faster R-CNN, and inference boxes are scaled back to original space for submission.\n",
  "AGENT_INSTRUCTIONS.md": "# Agent Instructions\n\nThis repository is a clean V1 Faster R-CNN baseline for the AMIA Public Challenge 2026. Keep it reproducible, leaderboard-valid, and Colab-friendly.\n\nEvery agent working on this repository must:\n\n1. Read `README.md` before making changes.\n2. Read this file before making changes.\n3. Check the current project structure before editing.\n4. Keep the README consistent with the code.\n5. Keep `notebooks/LG_CXR_FRCNN_Colab.ipynb` synchronized with script commands.\n6. Keep `notebooks/LG_CXR_FRCNN_Colab_Standalone.ipynb` synchronized with project files and script commands.\n7. Update README, this file, and both notebooks when any script command changes.\n8. Update README whenever behavior, commands, config, outputs, checkpoints, model choice, or submission logic changes.\n9. Never silently change the submission format.\n10. Never train class `14` as an object.\n11. Never output an empty `PredictionString`.\n12. Always run preflight before training.\n13. Prefer small, testable changes.\n14. Record important changes in the README changelog.\n15. Do not introduce YOLO or Ultralytics.\n16. Do not introduce RT-DETR for the first baseline.\n17. Do not over-engineer before a valid submission exists.\n18. Make every script runnable from the command line and from Colab.\n19. Keep config centralized in YAML.\n\nOperational notes:\n\n- Main detector: torchvision Faster R-CNN ResNet50-FPN.\n- Prefer explicit competition files and folders: `train/`, `test/`, `train.csv`, `test.csv`, `img_size.csv`, and `sample_submission.csv`.\n- Training IDs should come from the `train/` folder when available so folder-level negative images are not lost.\n- Test prediction may use `test.csv`, but final submission row order must come from `sample_submission.csv`.\n- Run `scripts/05_audit_dimensions.py` before changing `scanner.image_size` or `scanner.max_size`.\n- Treat `train.csv` boxes as original scan coordinates. Use `img_size.csv` to scale targets into PNG space for training and predictions back to original space for submission.\n- Detector labels: background is `0`; competition classes `0..13` map to internal labels `1..14`.\n- Class `14` is the no-finding fallback only and is ignored during detector training.\n- Empty test detections must become exactly `14 1.0 0 0 1 1`.\n- Primary output: `/kaggle/working/submission.csv` on Kaggle, or `WORK_DIR/submission.csv` in Colab.\n- Optional global and crop classifiers must never block V1 submission.\n",
  "configs/baseline_frcnn.yaml": "data_root: /kaggle/input/amia-public-challenge-2026\nwork_dir: /kaggle/working\nseed: 42\n\nsmoke_test: false\nsmoke_max_train_images: 500\nsmoke_max_val_images: 100\nsmoke_max_test_images: 100\n\nval_size: 0.2\n\nscanner:\n  model: fasterrcnn_resnet50_fpn\n  image_size: 800\n  max_size: 1200\n  epochs: 8\n  batch_size: 2\n  lr: 0.0002\n  weight_decay: 0.0001\n  num_workers: 2\n  amp: true\n  grad_clip: 5.0\n  best_ckpt: /kaggle/working/lgcxr_scanner_fasterrcnn_best.pth\n  last_ckpt: /kaggle/working/lgcxr_scanner_fasterrcnn_last.pth\n  resume: true\n\ninference:\n  conf_threshold: 0.03\n  nms_iou: 0.50\n  max_det_per_image: 100\n\nevaluation:\n  voc_iou_threshold: 0.4\n\nglobal_classifier:\n  enabled: false\n  model: resnet18\n  checkpoint: /kaggle/working/lgcxr_global_classifier_best.pth\n  image_size: 512\n\ncrop_classifier:\n  enabled: false\n  model: resnet18\n  checkpoint: /kaggle/working/lgcxr_crop_classifier_best.pth\n  image_size: 384\n  crop_margin: 0.4\n  max_crops_per_image: 30\n\nfusion:\n  scanner_weight: 1.0\n  global_weight: 0.0\n  crop_weight: 0.0\n  default_threshold: 0.03\n  per_class_thresholds: {}\n\nsubmission:\n  no_finding_class_id: 14\n  no_finding_string: \"14 1.0 0 0 1 1\"\n  output_path: /kaggle/working/submission.csv\n",
  "outputs/README.md": "# Outputs\n\nRuntime outputs are written here when `work_dir` points to the project output folder, especially in Colab.\n\nExpected files include:\n\n- `lgcxr_preflight_status.json`\n- `debug_preflight_annotation.png`\n- `lgcxr_scanner_fasterrcnn_best.pth`\n- `lgcxr_scanner_fasterrcnn_last.pth`\n- `lgcxr_fast_val_predictions.csv`\n- `lgcxr_fast_test_predictions.csv`\n- `lgcxr_scanner_training_summary.json`\n- `submission.csv`\n",
  "notebooks/README.md": "# Notebooks\n\n`LG_CXR_FRCNN_Colab.ipynb` is the runnable workflow for Colab when the project folder already exists in Drive or has been uploaded to Colab.\n\n`LG_CXR_FRCNN_Colab_Standalone.ipynb` is the copy/paste workflow for Colab when the user does not have the project folder yet. It bootstraps the project files into `/content/amia-lgcxr-frcnn`.\n\nBoth notebooks include a dimension-audit step:\n\n```bash\npython scripts/05_audit_dimensions.py --config configs/baseline_frcnn.yaml\n```\n\nRun it before changing `scanner.image_size` or `scanner.max_size`.\n\nThe notebooks rely on the project's coordinate-space fix: `train.csv` boxes are original scan coordinates, `img_size.csv` gives original dimensions, training targets are scaled into PNG space, and predictions are scaled back to original space for submission.\n\nKeep both notebooks synchronized with README commands and script arguments.\n",
  "src/__init__.py": "\"\"\"LG-CXR Faster R-CNN baseline package.\"\"\"\n",
  "src/data/__init__.py": "\"\"\"Data utilities for the LG-CXR Faster R-CNN baseline.\"\"\"\n",
  "src/data/columns.py": "\"\"\"Column inference and annotation normalization.\"\"\"\n\nfrom __future__ import annotations\n\nfrom dataclasses import asdict, dataclass\nfrom typing import Iterable\n\nimport numpy as np\nimport pandas as pd\n\nfrom src.utils.boxes import valid_box_mask, xywh_to_xyxy\n\n\n@dataclass(frozen=True)\nclass DetectionColumns:\n    image_id: str\n    class_id: str\n    xmin: str\n    ymin: str\n    xmax: str | None = None\n    ymax: str | None = None\n    width: str | None = None\n    height: str | None = None\n\n    @property\n    def uses_xywh(self) -> bool:\n        return self.width is not None and self.height is not None\n\n    def to_dict(self) -> dict:\n        return asdict(self)\n\n\n@dataclass(frozen=True)\nclass ImageSizeColumns:\n    image_id: str\n    width: str\n    height: str\n\n    def to_dict(self) -> dict:\n        return asdict(self)\n\n\ndef normalize_name(name: str) -> str:\n    return \"\".join(ch for ch in str(name).lower() if ch.isalnum())\n\n\ndef _find_column(columns: Iterable[str], candidates: Iterable[str]) -> str | None:\n    normalized = {normalize_name(col): col for col in columns}\n    for candidate in candidates:\n        key = normalize_name(candidate)\n        if key in normalized:\n            return normalized[key]\n    return None\n\n\ndef infer_detection_columns(df: pd.DataFrame) -> DetectionColumns:\n    columns = list(df.columns)\n\n    image_col = _find_column(\n        columns,\n        [\n            \"image_id\",\n            \"imageid\",\n            \"id\",\n            \"image\",\n            \"filename\",\n            \"file_name\",\n            \"image_name\",\n            \"image_path\",\n            \"path\",\n            \"study_id\",\n        ],\n    )\n    class_col = _find_column(\n        columns,\n        [\n            \"class_id\",\n            \"classid\",\n            \"class\",\n            \"label\",\n            \"target\",\n            \"category_id\",\n            \"category\",\n        ],\n    )\n    xmin = _find_column(columns, [\"xmin\", \"x_min\", \"x1\", \"left\"])\n    ymin = _find_column(columns, [\"ymin\", \"y_min\", \"y1\", \"top\"])\n    xmax = _find_column(columns, [\"xmax\", \"x_max\", \"x2\", \"right\"])\n    ymax = _find_column(columns, [\"ymax\", \"y_max\", \"y2\", \"bottom\"])\n    x = _find_column(columns, [\"x\", \"bbox_x\"])\n    y = _find_column(columns, [\"y\", \"bbox_y\"])\n    width = _find_column(columns, [\"width\", \"w\", \"bbox_width\"])\n    height = _find_column(columns, [\"height\", \"h\", \"bbox_height\"])\n\n    if image_col is None:\n        raise ValueError(f\"Could not infer image id column from: {columns}\")\n    if class_col is None:\n        raise ValueError(f\"Could not infer class id column from: {columns}\")\n    if xmin and ymin and xmax and ymax:\n        return DetectionColumns(image_col, class_col, xmin, ymin, xmax=xmax, ymax=ymax)\n    if x and y and width and height:\n        return DetectionColumns(image_col, class_col, x, y, width=width, height=height)\n\n    raise ValueError(\n        \"Could not infer bounding box columns. Expected xmin/ymin/xmax/ymax or x/y/width/height.\"\n    )\n\n\ndef infer_image_id_column(df: pd.DataFrame) -> str:\n    image_col = _find_column(\n        list(df.columns),\n        [\n            \"image_id\",\n            \"imageid\",\n            \"id\",\n            \"image\",\n            \"filename\",\n            \"file_name\",\n            \"image_name\",\n            \"image_path\",\n            \"path\",\n            \"study_id\",\n        ],\n    )\n    if image_col is None:\n        raise ValueError(f\"Could not infer image id column from: {list(df.columns)}\")\n    return image_col\n\n\ndef infer_image_size_columns(df: pd.DataFrame) -> ImageSizeColumns:\n    columns = list(df.columns)\n    image_col = infer_image_id_column(df)\n    width = _find_column(columns, [\"dim1\", \"width\", \"w\", \"image_width\", \"img_width\", \"cols\", \"columns\"])\n    height = _find_column(columns, [\"dim0\", \"height\", \"h\", \"image_height\", \"img_height\", \"rows\"])\n    if width is None or height is None:\n        raise ValueError(f\"Could not infer image size columns from: {columns}\")\n    return ImageSizeColumns(image_col, width, height)\n\n\ndef extract_image_ids(df: pd.DataFrame, image_col: str) -> list[str]:\n    return [str(v) for v in df[image_col].dropna().astype(str).unique().tolist()]\n\n\ndef extract_boxes_and_labels(df: pd.DataFrame, columns: DetectionColumns) -> tuple[np.ndarray, np.ndarray]:\n    if len(df) == 0:\n        return np.zeros((0, 4), dtype=float), np.zeros((0,), dtype=int)\n\n    if columns.uses_xywh:\n        boxes = df[[columns.xmin, columns.ymin, columns.width, columns.height]].apply(\n            pd.to_numeric, errors=\"coerce\"\n        ).to_numpy(dtype=float)\n        boxes = xywh_to_xyxy(boxes)\n    else:\n        boxes = df[[columns.xmin, columns.ymin, columns.xmax, columns.ymax]].apply(\n            pd.to_numeric, errors=\"coerce\"\n        ).to_numpy(dtype=float)\n\n    labels = pd.to_numeric(df[columns.class_id], errors=\"coerce\").to_numpy()\n    return boxes, labels\n\n\ndef object_rows(df: pd.DataFrame, columns: DetectionColumns, no_finding_class_id: int = 14) -> pd.DataFrame:\n    labels = pd.to_numeric(df[columns.class_id], errors=\"coerce\")\n    return df[(labels.notna()) & (labels.astype(int) != int(no_finding_class_id))].copy()\n\n\ndef validate_annotation_rows(\n    df: pd.DataFrame,\n    columns: DetectionColumns,\n    no_finding_class_id: int = 14,\n) -> dict:\n    obj = object_rows(df, columns, no_finding_class_id)\n    boxes, labels = extract_boxes_and_labels(obj, columns)\n    valid_box = valid_box_mask(boxes)\n    finite_label = np.isfinite(labels)\n    integer_label = np.zeros_like(finite_label, dtype=bool)\n    integer_label[finite_label] = labels[finite_label] == labels[finite_label].astype(int)\n    valid_label = finite_label & integer_label & (labels >= 0) & (labels <= 13)\n    valid = valid_box & valid_label\n    no_finding_rows = int((pd.to_numeric(df[columns.class_id], errors=\"coerce\") == no_finding_class_id).sum())\n    return {\n        \"rows\": int(len(df)),\n        \"object_rows\": int(len(obj)),\n        \"no_finding_rows\": no_finding_rows,\n        \"valid_object_rows\": int(valid.sum()),\n        \"invalid_object_rows\": int((~valid).sum()),\n    }\n",
  "src/data/dataset.py": "\"\"\"Torch datasets for Faster R-CNN training and inference.\"\"\"\n\nfrom __future__ import annotations\n\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom PIL import Image\nfrom torch.utils.data import Dataset\nfrom torchvision.transforms import functional as F\n\nfrom src.utils.boxes import clip_boxes_to_image, valid_box_mask\n\nfrom .columns import DetectionColumns, extract_boxes_and_labels, object_rows\nfrom .image_paths import image_id_variants, resolve_image_path\nfrom .image_sizes import SizeMap, scale_original_boxes_to_png\n\n\nclass CXRDetectionDataset(Dataset):\n    def __init__(\n        self,\n        annotations: pd.DataFrame,\n        image_ids: list[str],\n        image_index: dict[str, Path],\n        columns: DetectionColumns,\n        no_finding_class_id: int = 14,\n        original_size_map: SizeMap | None = None,\n    ) -> None:\n        self.annotations = annotations.copy()\n        self.image_ids = [str(v) for v in image_ids]\n        self.image_index = image_index\n        self.columns = columns\n        self.no_finding_class_id = int(no_finding_class_id)\n        self.original_size_map = original_size_map or {}\n        self.rows_by_image = {\n            variant: group.copy()\n            for image_id, group in self.annotations.groupby(self.columns.image_id, sort=False)\n            for variant in image_id_variants(str(image_id))\n        }\n\n    def __len__(self) -> int:\n        return len(self.image_ids)\n\n    def __getitem__(self, idx: int):\n        image_id = self.image_ids[idx]\n        path = resolve_image_path(image_id, self.image_index)\n        if path is None:\n            raise FileNotFoundError(f\"Could not resolve image path for image_id={image_id}\")\n\n        image = Image.open(path).convert(\"RGB\")\n        width, height = image.size\n        rows = self._lookup_rows(image_id)\n\n        if rows is None or len(rows) == 0:\n            boxes = np.zeros((0, 4), dtype=np.float32)\n            labels = np.zeros((0,), dtype=np.int64)\n        else:\n            rows = object_rows(rows, self.columns, self.no_finding_class_id)\n            boxes, labels = extract_boxes_and_labels(rows, self.columns)\n            boxes, _ = scale_original_boxes_to_png(\n                boxes,\n                image_id=image_id,\n                size_map=self.original_size_map,\n                png_width=width,\n                png_height=height,\n            )\n            boxes = clip_boxes_to_image(boxes, width, height)\n            keep = valid_box_mask(boxes) & (labels >= 0) & (labels <= 13)\n            boxes = boxes[keep].astype(np.float32)\n            labels = labels[keep].astype(np.int64) + 1\n\n        image_tensor = F.to_tensor(image)\n        boxes_tensor = torch.as_tensor(boxes, dtype=torch.float32)\n        labels_tensor = torch.as_tensor(labels, dtype=torch.int64)\n        area = (\n            (boxes_tensor[:, 2] - boxes_tensor[:, 0]) * (boxes_tensor[:, 3] - boxes_tensor[:, 1])\n            if boxes_tensor.numel()\n            else torch.zeros((0,), dtype=torch.float32)\n        )\n        target = {\n            \"boxes\": boxes_tensor.reshape(-1, 4),\n            \"labels\": labels_tensor,\n            \"image_id\": torch.tensor([idx], dtype=torch.int64),\n            \"area\": area,\n            \"iscrowd\": torch.zeros((len(labels_tensor),), dtype=torch.int64),\n        }\n        return image_tensor, target\n\n    def _lookup_rows(self, image_id: str) -> pd.DataFrame | None:\n        for variant in image_id_variants(image_id):\n            rows = self.rows_by_image.get(variant)\n            if rows is not None:\n                return rows\n        return None\n\n\nclass CXRImageDataset(Dataset):\n    def __init__(self, image_ids: list[str], image_index: dict[str, Path]) -> None:\n        self.image_ids = [str(v) for v in image_ids]\n        self.image_index = image_index\n\n    def __len__(self) -> int:\n        return len(self.image_ids)\n\n    def __getitem__(self, idx: int):\n        image_id = self.image_ids[idx]\n        path = resolve_image_path(image_id, self.image_index)\n        if path is None:\n            raise FileNotFoundError(f\"Could not resolve image path for image_id={image_id}\")\n        image = Image.open(path).convert(\"RGB\")\n        width, height = image.size\n        return F.to_tensor(image), {\"image_id\": image_id, \"path\": str(path), \"width\": width, \"height\": height}\n",
  "src/data/dimension_audit.py": "\"\"\"Dimension and bounding-box audit for the competition dataset.\"\"\"\n\nfrom __future__ import annotations\n\nimport json\nfrom collections import Counter\nfrom pathlib import Path\nfrom typing import Iterable\n\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image\n\nfrom src.utils.boxes import valid_box_mask\nfrom src.utils.config import ensure_work_dir\n\nfrom .columns import (\n    DetectionColumns,\n    extract_boxes_and_labels,\n    infer_detection_columns,\n    infer_image_id_column,\n    infer_image_size_columns,\n    object_rows,\n)\nfrom .discovery import discover_dataset, load_csv\nfrom .image_paths import build_image_index, image_id_variants, image_ids_from_paths, resolve_image_path\nfrom .image_sizes import build_image_size_map, lookup_original_size, scale_original_boxes_to_png\n\n\ndef run_dimension_audit(config: dict, max_images: int = 0) -> dict:\n    work_dir = ensure_work_dir(config)\n    no_finding_class_id = int(config[\"submission\"][\"no_finding_class_id\"])\n    scanner_cfg = config[\"scanner\"]\n    image_size = int(scanner_cfg[\"image_size\"])\n    max_size = int(scanner_cfg.get(\"max_size\", round(image_size * 1.5)))\n\n    discovery = discover_dataset(config[\"data_root\"])\n    if discovery.train_csv is None:\n        raise RuntimeError(\"Cannot audit dimensions without train.csv\")\n\n    train_df = load_csv(discovery.train_csv)\n    columns = infer_detection_columns(train_df)\n    img_size_report = _audit_img_size_csv(discovery.img_size_csv)\n\n    train_paths = discovery.train_image_paths or []\n    test_paths = discovery.test_image_paths or []\n    train_image_dims = _audit_pil_dimensions(train_paths, max_images=max_images)\n    test_image_dims = _audit_pil_dimensions(test_paths, max_images=max_images)\n\n    size_map = img_size_report.get(\"_size_map\", {})\n    png_size_map = _size_map_from_pil(train_image_dims.get(\"_records\", []), test_image_dims.get(\"_records\", []))\n    if not size_map:\n        size_map = png_size_map\n\n    box_report = _audit_boxes(\n        train_df=train_df,\n        columns=columns,\n        size_map=size_map,\n        no_finding_class_id=no_finding_class_id,\n        image_size=image_size,\n        max_size=max_size,\n        png_size_map=png_size_map,\n    )\n\n    train_folder_ids = image_ids_from_paths(train_paths) if train_paths else []\n    train_csv_ids = [str(v) for v in train_df[columns.image_id].dropna().astype(str).unique().tolist()]\n    alignment_report = {\n        \"train_folder_images\": len(train_folder_ids),\n        \"train_csv_unique_ids\": len(train_csv_ids),\n        \"train_csv_ids_missing_from_train_folder\": _count_missing_ids(train_csv_ids, build_image_index(train_paths)),\n    }\n\n    test_report = {}\n    if discovery.test_csv is not None:\n        test_df = load_csv(discovery.test_csv)\n        try:\n            test_col = infer_image_id_column(test_df)\n            test_csv_ids = [str(v) for v in test_df[test_col].dropna().astype(str).unique().tolist()]\n            test_report = {\n                \"test_csv_path\": str(discovery.test_csv),\n                \"test_csv_image_column\": test_col,\n                \"test_csv_unique_ids\": len(test_csv_ids),\n                \"test_csv_ids_missing_from_test_folder\": _count_missing_ids(test_csv_ids, build_image_index(test_paths)),\n            }\n        except Exception as exc:\n            test_report = {\"test_csv_path\": str(discovery.test_csv), \"error\": str(exc)}\n\n    report = {\n        \"data_root\": str(discovery.root),\n        \"train_csv\": str(discovery.train_csv),\n        \"test_csv\": str(discovery.test_csv) if discovery.test_csv else None,\n        \"img_size_csv\": str(discovery.img_size_csv) if discovery.img_size_csv else None,\n        \"sample_submission\": str(discovery.sample_submission) if discovery.sample_submission else None,\n        \"scanner_resize_config\": {\n            \"image_size_min_side\": image_size,\n            \"max_size_long_side\": max_size,\n            \"meaning\": \"torchvision detection transform resizes the shortest side to image_size, capped by max_size on the longest side\",\n        },\n        \"img_size_csv_audit\": _without_private(img_size_report),\n        \"train_image_file_audit\": _without_private(train_image_dims),\n        \"test_image_file_audit\": _without_private(test_image_dims),\n        \"csv_folder_alignment\": alignment_report,\n        \"test_csv_audit\": test_report,\n        \"train_box_audit\": box_report,\n        \"recommendation\": _recommend_resize(box_report, image_size, max_size),\n    }\n\n    output_path = work_dir / \"lgcxr_dimension_audit.json\"\n    output_path.write_text(json.dumps(report, indent=2), encoding=\"utf-8\")\n    report[\"output_path\"] = str(output_path)\n    return report\n\n\ndef _audit_img_size_csv(path: Path | None) -> dict:\n    if path is None:\n        return {\"present\": False, \"_size_map\": {}}\n    df = load_csv(path)\n    columns = infer_image_size_columns(df)\n    widths = pd.to_numeric(df[columns.width], errors=\"coerce\").to_numpy(dtype=float)\n    heights = pd.to_numeric(df[columns.height], errors=\"coerce\").to_numpy(dtype=float)\n    records = []\n    size_map = build_image_size_map(df, columns)\n    for _, row in df.iterrows():\n        try:\n            width = float(row[columns.width])\n            height = float(row[columns.height])\n        except Exception:\n            continue\n        if width <= 0 or height <= 0:\n            continue\n        image_id = str(row[columns.image_id])\n        records.append({\"image_id\": image_id, \"width\": width, \"height\": height})\n    return {\n        \"present\": True,\n        \"path\": str(path),\n        \"columns\": columns.to_dict(),\n        \"rows\": int(len(df)),\n        \"valid_rows\": int(len(records)),\n        \"width\": _numeric_summary(widths),\n        \"height\": _numeric_summary(heights),\n        \"aspect_ratio\": _numeric_summary(widths / np.maximum(heights, 1e-9)),\n        \"common_sizes\": _common_sizes([(r[\"width\"], r[\"height\"]) for r in records]),\n        \"_size_map\": size_map,\n    }\n\n\ndef _audit_pil_dimensions(paths: list[Path], max_images: int = 0) -> dict:\n    selected = paths if max_images <= 0 else paths[:max_images]\n    records = []\n    failures = []\n    for path in selected:\n        try:\n            with Image.open(path) as image:\n                width, height = image.size\n            records.append({\"image_id\": path.stem, \"path\": str(path), \"width\": float(width), \"height\": float(height)})\n        except Exception as exc:\n            failures.append({\"path\": str(path), \"error\": str(exc)})\n    widths = np.asarray([r[\"width\"] for r in records], dtype=float)\n    heights = np.asarray([r[\"height\"] for r in records], dtype=float)\n    return {\n        \"images_available\": int(len(paths)),\n        \"images_checked\": int(len(selected)),\n        \"valid_images\": int(len(records)),\n        \"failures\": failures[:10],\n        \"width\": _numeric_summary(widths),\n        \"height\": _numeric_summary(heights),\n        \"aspect_ratio\": _numeric_summary(widths / np.maximum(heights, 1e-9)) if len(records) else {},\n        \"common_sizes\": _common_sizes([(r[\"width\"], r[\"height\"]) for r in records]),\n        \"_records\": records,\n    }\n\n\ndef _audit_boxes(\n    train_df: pd.DataFrame,\n    columns: DetectionColumns,\n    size_map: dict[str, tuple[float, float]],\n    no_finding_class_id: int,\n    image_size: int,\n    max_size: int,\n    png_size_map: dict[str, tuple[float, float]],\n) -> dict:\n    object_df = object_rows(train_df, columns, no_finding_class_id)\n    boxes, labels = extract_boxes_and_labels(object_df, columns)\n    valid = valid_box_mask(boxes) & np.isfinite(labels) & (labels >= 0) & (labels <= 13)\n    boxes = boxes[valid]\n    labels = labels[valid].astype(int)\n    image_ids = object_df.loc[valid, columns.image_id].astype(str).tolist()\n\n    widths = boxes[:, 2] - boxes[:, 0] if len(boxes) else np.asarray([])\n    heights = boxes[:, 3] - boxes[:, 1] if len(boxes) else np.asarray([])\n    areas = widths * heights\n    class_counts = Counter(int(v) for v in labels.tolist())\n\n    rel_widths = []\n    rel_heights = []\n    rel_areas = []\n    png_box_widths = []\n    png_box_heights = []\n    resized_widths = []\n    resized_heights = []\n    missing_size = 0\n    missing_png_size = 0\n    outside_image = 0\n    normalized_like = 0\n    original_to_png_scaled = 0\n\n    for image_id, box, bw, bh, area in zip(image_ids, boxes, widths, heights, areas):\n        size = lookup_original_size(image_id, size_map)\n        if size is None:\n            missing_size += 1\n            continue\n        original_width, original_height = size\n        x1, y1, x2, y2 = [float(v) for v in box]\n        if max(x1, y1, x2, y2) <= 1.5:\n            normalized_like += 1\n        if x1 < 0 or y1 < 0 or x2 > original_width or y2 > original_height:\n            outside_image += 1\n        rel_widths.append(float(bw / max(original_width, 1e-9)))\n        rel_heights.append(float(bh / max(original_height, 1e-9)))\n        rel_areas.append(float(area / max(original_width * original_height, 1e-9)))\n\n        png_size = lookup_original_size(image_id, png_size_map)\n        if png_size is None:\n            missing_png_size += 1\n            continue\n        png_width, png_height = png_size\n        png_boxes, did_scale = scale_original_boxes_to_png(\n            np.asarray([box], dtype=float),\n            image_id=image_id,\n            size_map=size_map,\n            png_width=png_width,\n            png_height=png_height,\n        )\n        if did_scale:\n            original_to_png_scaled += 1\n        png_bw = float(png_boxes[0, 2] - png_boxes[0, 0])\n        png_bh = float(png_boxes[0, 3] - png_boxes[0, 1])\n        png_box_widths.append(png_bw)\n        png_box_heights.append(png_bh)\n        scale = _torchvision_resize_scale(png_width, png_height, image_size, max_size)\n        resized_widths.append(float(png_bw * scale))\n        resized_heights.append(float(png_bh * scale))\n\n    return {\n        \"train_rows\": int(len(train_df)),\n        \"object_rows_excluding_class_14\": int(len(object_df)),\n        \"valid_object_boxes\": int(len(boxes)),\n        \"class_counts\": {str(k): int(v) for k, v in sorted(class_counts.items())},\n        \"box_width_px\": _numeric_summary(widths),\n        \"box_height_px\": _numeric_summary(heights),\n        \"box_area_px\": _numeric_summary(areas),\n        \"box_width_fraction_of_image\": _numeric_summary(np.asarray(rel_widths, dtype=float)),\n        \"box_height_fraction_of_image\": _numeric_summary(np.asarray(rel_heights, dtype=float)),\n        \"box_area_fraction_of_image\": _numeric_summary(np.asarray(rel_areas, dtype=float)),\n        \"box_width_in_png_space_px\": _numeric_summary(np.asarray(png_box_widths, dtype=float)),\n        \"box_height_in_png_space_px\": _numeric_summary(np.asarray(png_box_heights, dtype=float)),\n        \"box_width_after_current_resize_px\": _numeric_summary(np.asarray(resized_widths, dtype=float)),\n        \"box_height_after_current_resize_px\": _numeric_summary(np.asarray(resized_heights, dtype=float)),\n        \"rows_missing_image_size\": int(missing_size),\n        \"rows_missing_png_size\": int(missing_png_size),\n        \"boxes_outside_declared_image\": int(outside_image),\n        \"normalized_coordinate_like_boxes\": int(normalized_like),\n        \"boxes_scaled_original_to_png\": int(original_to_png_scaled),\n    }\n\n\ndef _torchvision_resize_scale(width: float, height: float, image_size: int, max_size: int) -> float:\n    min_original = min(width, height)\n    max_original = max(width, height)\n    scale = float(image_size) / max(min_original, 1e-9)\n    if max_original * scale > max_size:\n        scale = float(max_size) / max(max_original, 1e-9)\n    return scale\n\n\ndef _recommend_resize(box_report: dict, image_size: int, max_size: int) -> dict:\n    q = box_report.get(\"box_width_after_current_resize_px\", {}).get(\"quantiles\", {})\n    p10_width = q.get(\"p10\")\n    p25_width = q.get(\"p25\")\n    p50_width = q.get(\"p50\")\n    outside = int(box_report.get(\"boxes_outside_declared_image\", 0))\n    normalized_like = int(box_report.get(\"normalized_coordinate_like_boxes\", 0))\n\n    notes = []\n    suggested = {\"image_size\": image_size, \"max_size\": max_size}\n    if normalized_like > 0:\n        notes.append(\"Some boxes look normalized; inspect CSV columns before training.\")\n    if outside > 0:\n        notes.append(\"Some boxes exceed declared image dimensions; inspect clipping or coordinate format.\")\n    if p10_width is not None and p10_width < 8:\n        suggested = {\"image_size\": 1200, \"max_size\": 1800}\n        notes.append(\"Small boxes become very small after resizing; try 1200/1800 if GPU memory allows.\")\n    elif p25_width is not None and p25_width < 12:\n        suggested = {\"image_size\": 1024, \"max_size\": 1536}\n        notes.append(\"Many boxes are small after resizing; try 1024/1536 as the next experiment.\")\n    elif p50_width is not None and p50_width > 80:\n        notes.append(\"Current 800/1200 resize likely preserves enough box detail for V1.\")\n    else:\n        notes.append(\"Current 800/1200 resize is a reasonable V1 baseline; tune after first validation result.\")\n\n    return {\n        \"current\": {\"image_size\": image_size, \"max_size\": max_size},\n        \"suggested_next_experiment\": suggested,\n        \"notes\": notes,\n    }\n\n\ndef _size_map_from_pil(*record_groups: Iterable[dict]) -> dict[str, tuple[float, float]]:\n    size_map = {}\n    for records in record_groups:\n        for record in records:\n            for variant in image_id_variants(record[\"image_id\"]):\n                size_map[variant] = (float(record[\"width\"]), float(record[\"height\"]))\n    return size_map\n\n\ndef _count_missing_ids(image_ids: list[str], image_index: dict) -> int:\n    return sum(resolve_image_path(image_id, image_index) is None for image_id in image_ids)\n\n\ndef _numeric_summary(values: np.ndarray) -> dict:\n    values = np.asarray(values, dtype=float)\n    values = values[np.isfinite(values)]\n    if values.size == 0:\n        return {\"count\": 0}\n    quantiles = np.percentile(values, [0, 1, 5, 10, 25, 50, 75, 90, 95, 99, 100])\n    return {\n        \"count\": int(values.size),\n        \"mean\": float(np.mean(values)),\n        \"std\": float(np.std(values)),\n        \"quantiles\": {\n            \"min\": float(quantiles[0]),\n            \"p01\": float(quantiles[1]),\n            \"p05\": float(quantiles[2]),\n            \"p10\": float(quantiles[3]),\n            \"p25\": float(quantiles[4]),\n            \"p50\": float(quantiles[5]),\n            \"p75\": float(quantiles[6]),\n            \"p90\": float(quantiles[7]),\n            \"p95\": float(quantiles[8]),\n            \"p99\": float(quantiles[9]),\n            \"max\": float(quantiles[10]),\n        },\n    }\n\n\ndef _common_sizes(sizes: Iterable[tuple[float, float]], limit: int = 10) -> list[dict]:\n    counter = Counter((int(round(w)), int(round(h))) for w, h in sizes)\n    return [{\"width\": w, \"height\": h, \"count\": count} for (w, h), count in counter.most_common(limit)]\n\n\ndef _without_private(report: dict) -> dict:\n    return {key: value for key, value in report.items() if not key.startswith(\"_\")}\n",
  "src/data/discovery.py": "\"\"\"Dataset discovery helpers.\"\"\"\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\nimport pandas as pd\n\nfrom .columns import infer_detection_columns\nfrom .image_paths import list_image_files\n\n\n@dataclass\nclass DatasetDiscovery:\n    root: Path\n    csv_paths: list[Path]\n    train_csv: Path | None\n    test_csv: Path | None\n    img_size_csv: Path | None\n    sample_submission: Path | None\n    train_dir: Path | None\n    test_dir: Path | None\n    train_image_paths: list[Path]\n    test_image_paths: list[Path]\n    image_paths: list[Path]\n\n\ndef find_csv_files(root: str | Path) -> list[Path]:\n    root = Path(root)\n    if not root.exists():\n        return []\n    return sorted(root.rglob(\"*.csv\"))\n\n\ndef load_csv(path: str | Path, **kwargs) -> pd.DataFrame:\n    return pd.read_csv(path, **kwargs)\n\n\ndef identify_sample_submission(csv_paths: list[Path]) -> Path | None:\n    scored: list[tuple[int, Path]] = []\n    for path in csv_paths:\n        score = 0\n        name = path.name.lower()\n        if \"sample\" in name and \"submission\" in name:\n            score += 5\n        elif \"submission\" in name:\n            score += 2\n        try:\n            columns = [str(c).lower() for c in pd.read_csv(path, nrows=3).columns]\n        except Exception:\n            continue\n        if \"image_id\" in columns and \"predictionstring\" in columns:\n            score += 10\n        if score > 0:\n            scored.append((score, path))\n    return sorted(scored, reverse=True)[0][1] if scored else None\n\n\ndef identify_train_csv(csv_paths: list[Path], sample_submission: Path | None = None) -> Path | None:\n    scored: list[tuple[int, Path]] = []\n    for path in csv_paths:\n        if sample_submission and path.resolve() == sample_submission.resolve():\n            continue\n        score = 0\n        name = path.name.lower()\n        if \"train\" in name:\n            score += 4\n        if \"annotation\" in name or \"bbox\" in name or \"label\" in name:\n            score += 2\n        try:\n            sample = pd.read_csv(path, nrows=100)\n            infer_detection_columns(sample)\n            score += 10\n        except Exception:\n            continue\n        scored.append((score, path))\n    return sorted(scored, reverse=True)[0][1] if scored else None\n\n\ndef _existing_file(root: Path, name: str) -> Path | None:\n    path = root / name\n    return path if path.exists() and path.is_file() else None\n\n\ndef _existing_dir(root: Path, name: str) -> Path | None:\n    path = root / name\n    return path if path.exists() and path.is_dir() else None\n\n\ndef discover_dataset(root: str | Path) -> DatasetDiscovery:\n    root = Path(root)\n    csv_paths = find_csv_files(root)\n    sample_submission = _existing_file(root, \"sample_submission.csv\") or identify_sample_submission(csv_paths)\n    train_csv = _existing_file(root, \"train.csv\") or identify_train_csv(csv_paths, sample_submission)\n    test_csv = _existing_file(root, \"test.csv\")\n    img_size_csv = _existing_file(root, \"img_size.csv\")\n    train_dir = _existing_dir(root, \"train\")\n    test_dir = _existing_dir(root, \"test\")\n    train_image_paths = list_image_files(train_dir) if train_dir else []\n    test_image_paths = list_image_files(test_dir) if test_dir else []\n    image_paths = list_image_files(root)\n    return DatasetDiscovery(\n        root=root,\n        csv_paths=csv_paths,\n        train_csv=train_csv,\n        test_csv=test_csv,\n        img_size_csv=img_size_csv,\n        sample_submission=sample_submission,\n        train_dir=train_dir,\n        test_dir=test_dir,\n        train_image_paths=train_image_paths,\n        test_image_paths=test_image_paths,\n        image_paths=image_paths,\n    )\n",
  "src/data/image_paths.py": "\"\"\"Image file indexing and resolution.\"\"\"\n\nfrom __future__ import annotations\n\nfrom pathlib import Path\n\n\nIMAGE_EXTENSIONS = {\".png\", \".jpg\", \".jpeg\", \".bmp\", \".tif\", \".tiff\"}\n\n\ndef list_image_files(root: str | Path) -> list[Path]:\n    root = Path(root)\n    if not root.exists():\n        return []\n    return sorted(p for p in root.rglob(\"*\") if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)\n\n\ndef image_id_from_path(path: str | Path) -> str:\n    return Path(path).stem\n\n\ndef image_id_variants(image_id: str) -> set[str]:\n    raw = str(image_id).strip().strip('\"').strip(\"'\")\n    normalized = raw.replace(\"\\\\\", \"/\")\n    path = Path(normalized)\n    candidates = {\n        raw,\n        normalized,\n        path.name,\n        path.stem,\n    }\n    return {candidate.lower() for candidate in candidates if candidate}\n\n\ndef image_ids_from_paths(paths: list[Path]) -> list[str]:\n    seen: set[str] = set()\n    image_ids: list[str] = []\n    for path in sorted(paths):\n        image_id = image_id_from_path(path)\n        key = image_id.lower()\n        if key not in seen:\n            seen.add(key)\n            image_ids.append(image_id)\n    return image_ids\n\n\ndef build_image_index(paths: list[Path]) -> dict[str, Path]:\n    index: dict[str, Path] = {}\n    for path in paths:\n        keys = image_id_variants(str(path)) | image_id_variants(path.name) | image_id_variants(path.stem)\n        for key in keys:\n            index.setdefault(key, path)\n    return index\n\n\ndef resolve_image_path(image_id: str, image_index: dict[str, Path]) -> Path | None:\n    for candidate in image_id_variants(image_id):\n        if candidate in image_index:\n            return image_index[candidate]\n    return None\n",
  "src/data/image_sizes.py": "\"\"\"Original-image dimension helpers.\n\nThe competition train boxes are in original scan coordinate space. The PNG\nfiles used for model input may be resized, commonly 1024x1024. Training boxes\ntherefore need to be scaled from original space into PNG space, while inference\nboxes need to be scaled back for submission.\n\"\"\"\n\nfrom __future__ import annotations\n\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nfrom .columns import ImageSizeColumns, infer_image_size_columns\nfrom .image_paths import image_id_variants\n\n\nSizeMap = dict[str, tuple[float, float]]\n\n\ndef load_image_size_map(path: str | Path | None) -> tuple[SizeMap, ImageSizeColumns | None]:\n    if path is None:\n        return {}, None\n    path = Path(path)\n    if not path.exists():\n        return {}, None\n    df = pd.read_csv(path)\n    columns = infer_image_size_columns(df)\n    return build_image_size_map(df, columns), columns\n\n\ndef build_image_size_map(df: pd.DataFrame, columns: ImageSizeColumns) -> SizeMap:\n    size_map: SizeMap = {}\n    for _, row in df.iterrows():\n        try:\n            width = float(row[columns.width])\n            height = float(row[columns.height])\n        except Exception:\n            continue\n        if width <= 0 or height <= 0:\n            continue\n        for variant in image_id_variants(str(row[columns.image_id])):\n            size_map[variant] = (width, height)\n    return size_map\n\n\ndef lookup_original_size(image_id: str, size_map: SizeMap) -> tuple[float, float] | None:\n    for variant in image_id_variants(str(image_id)):\n        size = size_map.get(variant)\n        if size is not None:\n            return size\n    return None\n\n\ndef scale_boxes_between_sizes(\n    boxes: np.ndarray,\n    from_width: float,\n    from_height: float,\n    to_width: float,\n    to_height: float,\n) -> np.ndarray:\n    boxes = np.asarray(boxes, dtype=float).copy()\n    if boxes.size == 0:\n        return boxes.reshape(-1, 4)\n    if from_width <= 0 or from_height <= 0:\n        return boxes\n    scale_x = float(to_width) / float(from_width)\n    scale_y = float(to_height) / float(from_height)\n    boxes[:, [0, 2]] *= scale_x\n    boxes[:, [1, 3]] *= scale_y\n    return boxes\n\n\ndef scale_original_boxes_to_png(\n    boxes: np.ndarray,\n    image_id: str,\n    size_map: SizeMap,\n    png_width: float,\n    png_height: float,\n) -> tuple[np.ndarray, bool]:\n    original_size = lookup_original_size(image_id, size_map)\n    if original_size is None:\n        return np.asarray(boxes, dtype=float), False\n    original_width, original_height = original_size\n    return (\n        scale_boxes_between_sizes(boxes, original_width, original_height, png_width, png_height),\n        not _same_size(original_width, original_height, png_width, png_height),\n    )\n\n\ndef scale_png_boxes_to_original(\n    boxes: np.ndarray,\n    image_id: str,\n    size_map: SizeMap,\n    png_width: float,\n    png_height: float,\n) -> tuple[np.ndarray, bool]:\n    original_size = lookup_original_size(image_id, size_map)\n    if original_size is None:\n        return np.asarray(boxes, dtype=float), False\n    original_width, original_height = original_size\n    return (\n        scale_boxes_between_sizes(boxes, png_width, png_height, original_width, original_height),\n        not _same_size(original_width, original_height, png_width, png_height),\n    )\n\n\ndef _same_size(width_a: float, height_a: float, width_b: float, height_b: float) -> bool:\n    return abs(float(width_a) - float(width_b)) < 1e-3 and abs(float(height_a) - float(height_b)) < 1e-3\n",
  "src/data/preflight.py": "\"\"\"Preflight checks that must pass before training.\"\"\"\n\nfrom __future__ import annotations\n\nimport json\nfrom pathlib import Path\n\nimport pandas as pd\nfrom PIL import Image\n\nfrom src.utils.config import ensure_work_dir\nfrom src.utils.visualization import draw_debug_annotation\n\nfrom .columns import (\n    extract_boxes_and_labels,\n    infer_detection_columns,\n    infer_image_id_column,\n    infer_image_size_columns,\n    object_rows,\n    validate_annotation_rows,\n)\nfrom .discovery import discover_dataset, load_csv\nfrom .image_paths import build_image_index, image_id_variants, image_ids_from_paths, resolve_image_path\nfrom .image_sizes import build_image_size_map, lookup_original_size, scale_original_boxes_to_png\nfrom .splits import sample_submission_image_ids\n\n\ndef run_preflight(config: dict) -> dict:\n    work_dir = ensure_work_dir(config)\n    root = Path(str(config[\"data_root\"]))\n    no_finding_class_id = int(config[\"submission\"][\"no_finding_class_id\"])\n    checks: list[dict] = []\n\n    def add(name: str, ok: bool, detail: str, critical: bool = True) -> None:\n        checks.append({\"name\": name, \"ok\": bool(ok), \"detail\": detail, \"critical\": bool(critical)})\n\n    add(\"dataset_exists\", root.exists(), str(root))\n    if not root.exists():\n        return _finish(work_dir, checks)\n\n    discovery = discover_dataset(root)\n    add(\"train_folder_found\", discovery.train_dir is not None, str(discovery.train_dir))\n    add(\"test_folder_found\", discovery.test_dir is not None, str(discovery.test_dir))\n    add(\"explicit_train_csv_found\", discovery.train_csv is not None and discovery.train_csv.name == \"train.csv\", str(discovery.train_csv))\n    add(\"explicit_test_csv_found\", discovery.test_csv is not None, str(discovery.test_csv))\n    add(\"explicit_img_size_csv_found\", discovery.img_size_csv is not None, str(discovery.img_size_csv), critical=False)\n    add(\n        \"explicit_sample_submission_found\",\n        discovery.sample_submission is not None and discovery.sample_submission.name == \"sample_submission.csv\",\n        str(discovery.sample_submission),\n    )\n    add(\"csv_files_found\", len(discovery.csv_paths) > 0, f\"{len(discovery.csv_paths)} CSV files\")\n    loaded_csvs = 0\n    for csv_path in discovery.csv_paths:\n        try:\n            pd.read_csv(csv_path, nrows=3)\n            loaded_csvs += 1\n        except Exception as exc:\n            add(\"csv_load_failed\", False, f\"{csv_path}: {exc}\")\n    add(\"csv_files_load\", loaded_csvs == len(discovery.csv_paths), f\"{loaded_csvs}/{len(discovery.csv_paths)} loaded\")\n\n    add(\"train_csv_identified\", discovery.train_csv is not None, str(discovery.train_csv))\n    add(\"sample_submission_identified\", discovery.sample_submission is not None, str(discovery.sample_submission))\n    add(\"train_images_found\", len(discovery.train_image_paths) > 0, f\"{len(discovery.train_image_paths)} train images\")\n    add(\"test_images_found\", len(discovery.test_image_paths) > 0, f\"{len(discovery.test_image_paths)} test images\")\n    add(\"images_found\", len(discovery.image_paths) > 0, f\"{len(discovery.image_paths)} total image files\")\n    if discovery.train_csv is None or discovery.sample_submission is None:\n        return _finish(work_dir, checks)\n\n    train_df = load_csv(discovery.train_csv)\n    sample_df = load_csv(discovery.sample_submission)\n    test_df = load_csv(discovery.test_csv) if discovery.test_csv is not None else None\n    img_size_df = load_csv(discovery.img_size_csv) if discovery.img_size_csv is not None else None\n\n    try:\n        columns = infer_detection_columns(train_df)\n        add(\"required_columns_inferred\", True, json.dumps(columns.to_dict()))\n    except Exception as exc:\n        add(\"required_columns_inferred\", False, str(exc))\n        return _finish(work_dir, checks)\n\n    add(\n        \"sample_submission_columns\",\n        list(sample_df.columns) == [\"image_id\", \"PredictionString\"],\n        f\"{list(sample_df.columns)}\",\n    )\n    if test_df is not None:\n        try:\n            test_image_col = infer_image_id_column(test_df)\n            add(\"test_csv_image_column_inferred\", True, test_image_col)\n        except Exception as exc:\n            add(\"test_csv_image_column_inferred\", False, str(exc))\n            test_image_col = None\n    else:\n        test_image_col = None\n\n    if img_size_df is not None:\n        try:\n            size_columns = infer_image_size_columns(img_size_df)\n            size_map = build_image_size_map(img_size_df, size_columns)\n            add(\"img_size_columns_inferred\", True, json.dumps(size_columns.to_dict()))\n            add(\"img_size_rows_loaded\", len(size_map) > 0, f\"{len(size_map)} image-size keys\", critical=False)\n        except Exception as exc:\n            size_columns = None\n            size_map = {}\n            add(\"img_size_columns_inferred\", False, str(exc), critical=False)\n    else:\n        size_columns = None\n        size_map = {}\n\n    validation = validate_annotation_rows(train_df, columns, no_finding_class_id)\n    add(\"bounding_boxes_valid\", validation[\"invalid_object_rows\"] == 0, json.dumps(validation))\n    if size_map:\n        add(\"boxes_within_img_size\", *_validate_boxes_against_size(train_df, columns, size_map, no_finding_class_id), critical=False)\n    add(\n        \"class_14_no_finding_only\",\n        True,\n        f\"{validation['no_finding_rows']} rows with class {no_finding_class_id} will be ignored for detector training\",\n    )\n\n    train_image_paths = discovery.train_image_paths or discovery.image_paths\n    test_image_paths = discovery.test_image_paths or discovery.image_paths\n    train_image_index = build_image_index(train_image_paths)\n    test_image_index = build_image_index(test_image_paths)\n\n    train_ids_from_csv = [str(v) for v in train_df[columns.image_id].dropna().astype(str).unique().tolist()]\n    train_ids = image_ids_from_paths(train_image_paths) if discovery.train_image_paths else train_ids_from_csv\n    test_ids = sample_submission_image_ids(sample_df)\n    test_ids_from_csv = (\n        [str(v) for v in test_df[test_image_col].dropna().astype(str).unique().tolist()]\n        if test_df is not None and test_image_col is not None\n        else []\n    )\n\n    train_resolved = [resolve_image_path(image_id, train_image_index) for image_id in train_ids]\n    train_csv_resolved = [resolve_image_path(image_id, train_image_index) for image_id in train_ids_from_csv]\n    test_resolved = [resolve_image_path(image_id, test_image_index) for image_id in test_ids]\n    add(\"train_folder_image_ids_used\", len(train_ids) > 0, f\"{len(train_ids)} train IDs from train/ folder\")\n    add(\"train_images_resolve\", all(p is not None for p in train_resolved), f\"{sum(p is not None for p in train_resolved)}/{len(train_ids)}\")\n    add(\"train_csv_rows_resolve_to_train_folder\", all(p is not None for p in train_csv_resolved), f\"{sum(p is not None for p in train_csv_resolved)}/{len(train_ids_from_csv)}\")\n    add(\"test_images_resolve\", all(p is not None for p in test_resolved), f\"{sum(p is not None for p in test_resolved)}/{len(test_ids)}\")\n    if test_ids_from_csv:\n        test_csv_resolved = [resolve_image_path(image_id, test_image_index) for image_id in test_ids_from_csv]\n        add(\"test_csv_rows_resolve_to_test_folder\", all(p is not None for p in test_csv_resolved), f\"{sum(p is not None for p in test_csv_resolved)}/{len(test_ids_from_csv)}\")\n        add(\"test_csv_matches_sample_ids\", _same_id_set(test_ids_from_csv, test_ids), f\"test.csv={len(test_ids_from_csv)} sample_submission={len(test_ids)}\", critical=False)\n\n    add(\"train_images_open\", *_open_sample(train_resolved, 50, \"train\"))\n    add(\"test_images_open\", *_open_sample(test_resolved, 20, \"test\"))\n    if size_map:\n        add(\n            \"coordinate_space_scaling_required\",\n            True,\n            _coordinate_space_summary(train_ids[:50], train_image_index, size_map),\n            critical=False,\n        )\n\n    debug_saved = False\n    debug_detail = \"No resolvable training image\"\n    for image_id in train_ids:\n        path = resolve_image_path(image_id, train_image_index)\n        if path is None:\n            continue\n        rows = train_df[train_df[columns.image_id].astype(str) == str(image_id)]\n        rows = object_rows(rows, columns, no_finding_class_id)\n        boxes, labels = extract_boxes_and_labels(rows, columns)\n        with Image.open(path) as image:\n            width, height = image.size\n        boxes, _ = scale_original_boxes_to_png(boxes, image_id, size_map, width, height)\n        debug_path = work_dir / \"debug_preflight_annotation.png\"\n        draw_debug_annotation(path, boxes[:10], labels[:10].astype(int).tolist(), debug_path)\n        debug_saved = True\n        debug_detail = str(debug_path)\n        break\n    add(\"debug_image_saved\", debug_saved, debug_detail)\n\n    return _finish(work_dir, checks)\n\n\ndef _open_sample(paths, max_count: int, label: str) -> tuple[bool, str]:\n    sample = [p for p in paths if p is not None][:max_count]\n    if not sample:\n        return False, f\"No {label} images resolved\"\n    opened = 0\n    failures = []\n    for path in sample:\n        try:\n            with Image.open(path) as image:\n                image.verify()\n            opened += 1\n        except Exception as exc:\n            failures.append(f\"{path}: {exc}\")\n    ok = opened == len(sample)\n    detail = f\"{opened}/{len(sample)} opened\"\n    if failures:\n        detail += f\"; first failure: {failures[0]}\"\n    return ok, detail\n\n\ndef _validate_boxes_against_size(train_df, columns, size_map, no_finding_class_id: int) -> tuple[bool, str]:\n    rows = object_rows(train_df, columns, no_finding_class_id)\n    if len(rows) == 0:\n        return True, \"No object rows to validate against img_size.csv\"\n    boxes, _ = extract_boxes_and_labels(rows, columns)\n    missing_size = 0\n    outside = 0\n    normalized_like = 0\n    checked = 0\n    for (_, row), box in zip(rows.iterrows(), boxes):\n        image_id = str(row[columns.image_id])\n        size = lookup_original_size(image_id, size_map)\n        if size is None:\n            missing_size += 1\n            continue\n        width, height = size\n        x1, y1, x2, y2 = [float(v) for v in box]\n        checked += 1\n        if max(x1, y1, x2, y2) <= 1.5:\n            normalized_like += 1\n        if x1 < 0 or y1 < 0 or x2 > width or y2 > height:\n            outside += 1\n    ok = outside == 0\n    detail = (\n        f\"checked={checked}, outside_image={outside}, missing_size={missing_size}, \"\n        f\"normalized_like={normalized_like}\"\n    )\n    return ok, detail\n\n\ndef _coordinate_space_summary(image_ids, image_index, size_map) -> str:\n    checked = 0\n    different = 0\n    examples = []\n    for image_id in image_ids:\n        path = resolve_image_path(image_id, image_index)\n        original_size = lookup_original_size(image_id, size_map)\n        if path is None or original_size is None:\n            continue\n        with Image.open(path) as image:\n            png_width, png_height = image.size\n        orig_width, orig_height = original_size\n        checked += 1\n        if abs(orig_width - png_width) > 1 or abs(orig_height - png_height) > 1:\n            different += 1\n            if len(examples) < 3:\n                examples.append(f\"{image_id}: original={int(orig_width)}x{int(orig_height)} png={png_width}x{png_height}\")\n    return f\"{different}/{checked} checked train images need original-to-PNG box scaling; examples: {examples}\"\n\n\ndef _same_id_set(left: list[str], right: list[str]) -> bool:\n    left_keys = {_canonical_id(v) for v in left}\n    right_keys = {_canonical_id(v) for v in right}\n    return left_keys == right_keys\n\n\ndef _canonical_id(value: str) -> str:\n    normalized = str(value).strip().strip('\"').strip(\"'\").replace(\"\\\\\", \"/\")\n    return Path(normalized).stem.lower()\n\n\ndef _finish(work_dir: Path, checks: list[dict]) -> dict:\n    ok = all(check[\"ok\"] for check in checks if check[\"critical\"])\n    result = {\"ok\": ok, \"checks\": checks}\n    status_path = work_dir / \"lgcxr_preflight_status.json\"\n    status_path.parent.mkdir(parents=True, exist_ok=True)\n    status_path.write_text(json.dumps(result, indent=2), encoding=\"utf-8\")\n    if not ok:\n        failed = [check for check in checks if check[\"critical\"] and not check[\"ok\"]]\n        details = \"; \".join(f\"{check['name']}: {check['detail']}\" for check in failed)\n        raise RuntimeError(f\"Preflight failed: {details}\")\n    return result\n",
  "src/data/splits.py": "\"\"\"Train/validation/test split helpers.\"\"\"\n\nfrom __future__ import annotations\n\nimport random\n\nimport pandas as pd\n\nfrom .columns import DetectionColumns, extract_image_ids\n\n\ndef make_train_val_split(\n    train_df: pd.DataFrame,\n    columns: DetectionColumns,\n    val_size: float,\n    seed: int,\n    all_image_ids: list[str] | None = None,\n    smoke_test: bool = False,\n    smoke_max_train_images: int = 500,\n    smoke_max_val_images: int = 100,\n) -> tuple[list[str], list[str]]:\n    image_ids = [str(v) for v in all_image_ids] if all_image_ids is not None else extract_image_ids(train_df, columns.image_id)\n    rng = random.Random(seed)\n    rng.shuffle(image_ids)\n\n    if len(image_ids) <= 1:\n        return image_ids, []\n\n    val_count = max(1, int(round(len(image_ids) * float(val_size))))\n    val_count = min(val_count, len(image_ids) - 1)\n    val_ids = image_ids[:val_count]\n    train_ids = image_ids[val_count:]\n\n    if smoke_test:\n        train_ids = train_ids[: int(smoke_max_train_images)]\n        val_ids = val_ids[: int(smoke_max_val_images)]\n\n    return train_ids, val_ids\n\n\ndef sample_submission_image_ids(sample_df: pd.DataFrame, smoke_test: bool = False, smoke_max_test_images: int = 100) -> list[str]:\n    if \"image_id\" not in sample_df.columns:\n        raise ValueError(\"Sample submission must contain image_id\")\n    image_ids = [str(v) for v in sample_df[\"image_id\"].astype(str).tolist()]\n    if smoke_test:\n        return image_ids[: int(smoke_max_test_images)]\n    return image_ids\n",
  "src/inference/__init__.py": "\"\"\"Inference helpers for the LG-CXR Faster R-CNN baseline.\"\"\"\n",
  "src/inference/fuse_predictions.py": "\"\"\"V1 prediction fusion placeholder.\"\"\"\n\nfrom __future__ import annotations\n\nimport pandas as pd\n\n\ndef fuse_v1(scanner_predictions: pd.DataFrame, *_, **__) -> pd.DataFrame:\n    \"\"\"Return scanner predictions unchanged for V1.\"\"\"\n    return scanner_predictions.copy()\n",
  "src/inference/make_submission.py": "\"\"\"Submission formatting and validation.\"\"\"\n\nfrom __future__ import annotations\n\nfrom pathlib import Path\n\nimport pandas as pd\n\nfrom src.data.image_paths import image_id_variants\n\n\nPREDICTION_COLUMNS = [\"image_id\", \"class_id\", \"confidence\", \"xmin\", \"ymin\", \"xmax\", \"ymax\"]\nSUBMISSION_COLUMNS = [\"image_id\", \"PredictionString\"]\n\n\ndef make_submission(\n    sample_submission: pd.DataFrame,\n    predictions: pd.DataFrame,\n    output_path: str | Path,\n    no_finding_string: str = \"14 1.0 0 0 1 1\",\n) -> pd.DataFrame:\n    if list(sample_submission.columns) != SUBMISSION_COLUMNS:\n        raise ValueError(f\"Sample submission columns must be exactly {SUBMISSION_COLUMNS}\")\n\n    if predictions.empty:\n        predictions = pd.DataFrame(columns=PREDICTION_COLUMNS)\n    missing = [col for col in PREDICTION_COLUMNS if col not in predictions.columns]\n    if missing:\n        raise ValueError(f\"Prediction CSV is missing columns: {missing}\")\n\n    predictions = predictions.copy()\n    predictions[\"image_id\"] = predictions[\"image_id\"].astype(str)\n    grouped = {}\n    for image_id, group in predictions.groupby(\"image_id\", sort=False):\n        sorted_group = group.sort_values(\"confidence\", ascending=False)\n        for variant in image_id_variants(str(image_id)):\n            grouped.setdefault(variant, sorted_group)\n\n    rows = []\n    for image_id in sample_submission[\"image_id\"].astype(str).tolist():\n        group = _lookup_prediction_group(grouped, image_id)\n        if group is None or len(group) == 0:\n            prediction_string = no_finding_string\n        else:\n            parts = []\n            for _, row in group.iterrows():\n                class_id = int(row[\"class_id\"])\n                if class_id < 0 or class_id > 13:\n                    continue\n                parts.extend(\n                    [\n                        str(class_id),\n                        _format_float(float(row[\"confidence\"])),\n                        _format_float(float(row[\"xmin\"])),\n                        _format_float(float(row[\"ymin\"])),\n                        _format_float(float(row[\"xmax\"])),\n                        _format_float(float(row[\"ymax\"])),\n                    ]\n                )\n            prediction_string = \" \".join(parts) if parts else no_finding_string\n        rows.append({\"image_id\": image_id, \"PredictionString\": prediction_string})\n\n    submission = pd.DataFrame(rows, columns=SUBMISSION_COLUMNS)\n    validate_submission(submission, sample_submission, no_finding_string)\n    output_path = Path(output_path)\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    submission.to_csv(output_path, index=False)\n    return submission\n\n\ndef validate_submission(\n    submission: pd.DataFrame,\n    sample_submission: pd.DataFrame,\n    no_finding_string: str = \"14 1.0 0 0 1 1\",\n) -> None:\n    if list(submission.columns) != SUBMISSION_COLUMNS:\n        raise AssertionError(f\"Submission columns must be exactly {SUBMISSION_COLUMNS}\")\n    if len(submission) != len(sample_submission):\n        raise AssertionError(\"Submission row count must equal sample submission row count\")\n    if submission.isna().any().any():\n        raise AssertionError(\"Submission must not contain NaN values\")\n\n    for prediction_string in submission[\"PredictionString\"].astype(str):\n        if not prediction_string.strip():\n            raise AssertionError(\"PredictionString must never be empty\")\n        tokens = prediction_string.split()\n        if len(tokens) % 6 != 0:\n            raise AssertionError(f\"PredictionString group length must be a multiple of 6: {prediction_string}\")\n\n        class_ids = [int(float(tokens[i])) for i in range(0, len(tokens), 6)]\n        if any(class_id < 0 or class_id > 14 for class_id in class_ids):\n            raise AssertionError(f\"Class IDs must be in 0..14: {prediction_string}\")\n        if 14 in class_ids and prediction_string != no_finding_string:\n            raise AssertionError(\"Class 14 may only appear alone as the exact no-finding fallback\")\n\n\ndef _lookup_prediction_group(grouped: dict, image_id: str):\n    for variant in image_id_variants(str(image_id)):\n        group = grouped.get(variant)\n        if group is not None:\n            return group\n    return None\n\n\ndef _format_float(value: float) -> str:\n    return f\"{value:.6g}\"\n",
  "src/inference/predict_crop.py": "\"\"\"Optional crop classifier inference placeholder.\"\"\"\n\nfrom __future__ import annotations\n\nfrom pathlib import Path\nimport warnings\n\n\ndef load_optional_crop_classifier(checkpoint_path: str | Path):\n    checkpoint_path = Path(checkpoint_path)\n    if not checkpoint_path.exists():\n        warnings.warn(f\"Crop classifier checkpoint missing; continuing without it: {checkpoint_path}\")\n        return None\n    raise NotImplementedError(\"Crop classifier inference is intentionally deferred for V1.\")\n",
  "src/inference/predict_global.py": "\"\"\"Optional global classifier inference placeholder.\"\"\"\n\nfrom __future__ import annotations\n\nfrom pathlib import Path\nimport warnings\n\n\ndef load_optional_global_classifier(checkpoint_path: str | Path):\n    checkpoint_path = Path(checkpoint_path)\n    if not checkpoint_path.exists():\n        warnings.warn(f\"Global classifier checkpoint missing; continuing without it: {checkpoint_path}\")\n        return None\n    raise NotImplementedError(\"Global classifier inference is intentionally deferred for V1.\")\n",
  "src/inference/predict_scanner.py": "\"\"\"Faster R-CNN prediction helpers.\"\"\"\n\nfrom __future__ import annotations\n\nfrom pathlib import Path\n\nimport pandas as pd\nimport torch\nfrom tqdm.auto import tqdm\n\nfrom src.utils.boxes import classwise_nms, clip_boxes_to_image\nfrom src.data.image_sizes import SizeMap, scale_png_boxes_to_original\n\n\ndef image_collate(batch):\n    images, metas = zip(*batch)\n    return list(images), list(metas)\n\n\n@torch.no_grad()\ndef predict_dataset(\n    model: torch.nn.Module,\n    loader,\n    device: torch.device,\n    conf_threshold: float,\n    nms_iou: float,\n    max_det_per_image: int,\n    original_size_map: SizeMap | None = None,\n) -> pd.DataFrame:\n    model.eval()\n    records = []\n\n    for images, metas in tqdm(loader, desc=\"predict\", leave=False):\n        images = [image.to(device) for image in images]\n        outputs = model(images)\n\n        for output, meta in zip(outputs, metas):\n            image_id = str(meta[\"image_id\"])\n            png_width = float(meta.get(\"width\", 0))\n            png_height = float(meta.get(\"height\", 0))\n            boxes = output[\"boxes\"].detach()\n            scores = output[\"scores\"].detach()\n            labels = output[\"labels\"].detach()\n\n            keep = (scores >= float(conf_threshold)) & (labels >= 1) & (labels <= 14)\n            boxes = boxes[keep]\n            scores = scores[keep]\n            labels = labels[keep]\n\n            if boxes.numel() > 0:\n                keep_indices = classwise_nms(boxes, scores, labels, float(nms_iou))\n                keep_indices = keep_indices[: int(max_det_per_image)]\n                boxes = boxes[keep_indices].cpu()\n                scores = scores[keep_indices].cpu()\n                labels = labels[keep_indices].cpu()\n                if original_size_map and png_width > 0 and png_height > 0:\n                    boxes_np, _ = scale_png_boxes_to_original(\n                        boxes.numpy(),\n                        image_id=image_id,\n                        size_map=original_size_map,\n                        png_width=png_width,\n                        png_height=png_height,\n                    )\n                    boxes_np = _clip_to_original_size(boxes_np, image_id, original_size_map)\n                    boxes = torch.as_tensor(boxes_np, dtype=torch.float32)\n\n            for box, score, label in zip(boxes, scores, labels):\n                x1, y1, x2, y2 = [float(v) for v in box.tolist()]\n                records.append(\n                    {\n                        \"image_id\": image_id,\n                        \"class_id\": int(label.item()) - 1,\n                        \"confidence\": float(score.item()),\n                        \"xmin\": x1,\n                        \"ymin\": y1,\n                        \"xmax\": x2,\n                        \"ymax\": y2,\n                    }\n                )\n\n    columns = [\"image_id\", \"class_id\", \"confidence\", \"xmin\", \"ymin\", \"xmax\", \"ymax\"]\n    return pd.DataFrame.from_records(records, columns=columns)\n\n\ndef save_predictions(df: pd.DataFrame, path: str | Path) -> Path:\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    df.to_csv(path, index=False)\n    return path\n\n\ndef _clip_to_original_size(boxes_np, image_id: str, original_size_map: SizeMap):\n    from src.data.image_sizes import lookup_original_size\n\n    original_size = lookup_original_size(image_id, original_size_map)\n    if original_size is None:\n        return boxes_np\n    width, height = original_size\n    return clip_boxes_to_image(boxes_np, int(round(width)), int(round(height)))\n",
  "src/models/__init__.py": "\"\"\"Model builders for the LG-CXR Faster R-CNN baseline.\"\"\"\n",
  "src/models/fasterrcnn.py": "\"\"\"Faster R-CNN model builder.\"\"\"\n\nfrom __future__ import annotations\n\nimport os\nimport warnings\nfrom pathlib import Path\nfrom urllib.parse import urlparse\n\nimport torch\nfrom torchvision.models.detection.faster_rcnn import FastRCNNPredictor\n\n\ndef build_fasterrcnn(num_classes: int, image_size: int = 800, max_size: int | None = None):\n    \"\"\"Build torchvision Faster R-CNN and replace the classification head.\n\n    `num_classes` includes the background class. For this competition V1 uses\n    15 classes: background plus pathology IDs 0..13 mapped to labels 1..14.\n    \"\"\"\n    import torchvision.models.detection as detection\n\n    builder = getattr(detection, \"fasterrcnn_resnet50_fpn_v2\", None)\n    weights_cls = getattr(detection, \"FasterRCNN_ResNet50_FPN_V2_Weights\", None)\n    if builder is None:\n        builder = getattr(detection, \"fasterrcnn_resnet50_fpn\")\n        weights_cls = getattr(detection, \"FasterRCNN_ResNet50_FPN_Weights\", None)\n\n    weights = _cached_or_allowed_weights(weights_cls)\n    if max_size is None:\n        max_size = int(round(image_size * 1.5))\n\n    kwargs = {\n        \"weights\": weights,\n        \"weights_backbone\": None,\n        \"min_size\": int(image_size),\n        \"max_size\": int(max_size),\n    }\n\n    try:\n        model = builder(**kwargs)\n    except TypeError:\n        model = builder(\n            pretrained=weights is not None,\n            pretrained_backbone=False,\n            min_size=int(image_size),\n            max_size=int(max_size),\n        )\n\n    in_features = model.roi_heads.box_predictor.cls_score.in_features\n    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)\n    return model\n\n\ndef _cached_or_allowed_weights(weights_cls):\n    if weights_cls is None:\n        return None\n    weights = weights_cls.DEFAULT\n    if os.environ.get(\"LGCXR_ALLOW_WEIGHT_DOWNLOAD\", \"0\") == \"1\":\n        return weights\n\n    filename = Path(urlparse(getattr(weights, \"url\", \"\")).path).name\n    if not filename:\n        return None\n    cached = Path(torch.hub.get_dir()) / \"checkpoints\" / filename\n    if cached.exists():\n        return weights\n\n    warnings.warn(\n        \"Torchvision pretrained detector weights were not found in the local cache. \"\n        \"Using randomly initialized detector weights. Set LGCXR_ALLOW_WEIGHT_DOWNLOAD=1 \"\n        \"to permit torchvision downloads when internet access is available.\",\n        RuntimeWarning,\n    )\n    return None\n",
  "src/models/resnet18_classifier.py": "\"\"\"Optional ResNet18 classifier builder for future global/crop models.\"\"\"\n\nfrom __future__ import annotations\n\nimport torch\n\n\ndef build_resnet18_classifier(num_classes: int, pretrained: bool = False) -> torch.nn.Module:\n    try:\n        import timm\n\n        return timm.create_model(\"resnet18\", pretrained=pretrained, num_classes=num_classes)\n    except Exception:\n        from torchvision import models\n\n        weights = models.ResNet18_Weights.DEFAULT if pretrained and hasattr(models, \"ResNet18_Weights\") else None\n        model = models.resnet18(weights=weights)\n        model.fc = torch.nn.Linear(model.fc.in_features, num_classes)\n        return model\n",
  "src/train/__init__.py": "\"\"\"Training helpers for the LG-CXR Faster R-CNN baseline.\"\"\"\n",
  "src/train/train_scanner.py": "\"\"\"Training helpers for the Faster R-CNN scanner.\"\"\"\n\nfrom __future__ import annotations\n\nfrom typing import Iterable\n\nimport torch\nfrom tqdm.auto import tqdm\n\nfrom src.utils.metrics_voc import voc_map\n\n\ndef detection_collate(batch):\n    images, targets = zip(*batch)\n    return list(images), list(targets)\n\n\ndef move_targets_to_device(targets: Iterable[dict], device: torch.device) -> list[dict]:\n    moved = []\n    for target in targets:\n        moved.append({key: value.to(device) if torch.is_tensor(value) else value for key, value in target.items()})\n    return moved\n\n\ndef make_optimizer(model: torch.nn.Module, lr: float, weight_decay: float) -> torch.optim.Optimizer:\n    params = [p for p in model.parameters() if p.requires_grad]\n    return torch.optim.AdamW(params, lr=float(lr), weight_decay=float(weight_decay))\n\n\ndef train_one_epoch(\n    model: torch.nn.Module,\n    loader,\n    optimizer: torch.optim.Optimizer,\n    device: torch.device,\n    amp: bool = True,\n    grad_clip: float | None = None,\n    scaler=None,\n    epoch: int = 0,\n) -> float:\n    model.train()\n    losses = []\n    amp_enabled = bool(amp and device.type == \"cuda\")\n    progress = tqdm(loader, desc=f\"train epoch {epoch}\", leave=False)\n\n    for images, targets in progress:\n        images = [image.to(device) for image in images]\n        targets = move_targets_to_device(targets, device)\n        optimizer.zero_grad(set_to_none=True)\n\n        with torch.cuda.amp.autocast(enabled=amp_enabled):\n            loss_dict = model(images, targets)\n            loss = sum(loss for loss in loss_dict.values())\n\n        if scaler is not None and amp_enabled:\n            scaler.scale(loss).backward()\n            if grad_clip:\n                scaler.unscale_(optimizer)\n                torch.nn.utils.clip_grad_norm_(model.parameters(), float(grad_clip))\n            scaler.step(optimizer)\n            scaler.update()\n        else:\n            loss.backward()\n            if grad_clip:\n                torch.nn.utils.clip_grad_norm_(model.parameters(), float(grad_clip))\n            optimizer.step()\n\n        loss_value = float(loss.detach().cpu().item())\n        losses.append(loss_value)\n        progress.set_postfix(loss=f\"{loss_value:.4f}\")\n\n    return float(sum(losses) / max(len(losses), 1))\n\n\n@torch.no_grad()\ndef evaluate_detector_map(\n    model: torch.nn.Module,\n    loader,\n    dataset,\n    device: torch.device,\n    iou_threshold: float = 0.4,\n    conf_threshold: float = 0.0,\n) -> dict:\n    model.eval()\n    ground_truth = []\n    predictions = []\n\n    for images, targets in tqdm(loader, desc=\"valid\", leave=False):\n        images = [image.to(device) for image in images]\n        outputs = model(images)\n        for output, target in zip(outputs, targets):\n            image_idx = int(target[\"image_id\"].item())\n            image_id = dataset.image_ids[image_idx]\n\n            gt_boxes = target[\"boxes\"].detach().cpu().numpy()\n            gt_labels = target[\"labels\"].detach().cpu().numpy()\n            for box, label in zip(gt_boxes, gt_labels):\n                if 1 <= int(label) <= 14:\n                    ground_truth.append(\n                        {\"image_id\": image_id, \"class_id\": int(label) - 1, \"box\": [float(v) for v in box]}\n                    )\n\n            boxes = output[\"boxes\"].detach().cpu().numpy()\n            labels = output[\"labels\"].detach().cpu().numpy()\n            scores = output[\"scores\"].detach().cpu().numpy()\n            for box, label, score in zip(boxes, labels, scores):\n                if float(score) < conf_threshold or int(label) < 1 or int(label) > 14:\n                    continue\n                predictions.append(\n                    {\n                        \"image_id\": image_id,\n                        \"class_id\": int(label) - 1,\n                        \"score\": float(score),\n                        \"box\": [float(v) for v in box],\n                    }\n                )\n\n    return voc_map(ground_truth, predictions, class_ids=range(14), iou_threshold=float(iou_threshold))\n",
  "src/utils/__init__.py": "\"\"\"General utilities for the LG-CXR Faster R-CNN baseline.\"\"\"\n",
  "src/utils/boxes.py": "\"\"\"Bounding box helpers.\"\"\"\n\nfrom __future__ import annotations\n\nimport numpy as np\n\n\ndef xywh_to_xyxy(boxes: np.ndarray) -> np.ndarray:\n    boxes = boxes.astype(float, copy=True)\n    boxes[:, 2] = boxes[:, 0] + boxes[:, 2]\n    boxes[:, 3] = boxes[:, 1] + boxes[:, 3]\n    return boxes\n\n\ndef clip_boxes_to_image(boxes: np.ndarray, width: int, height: int) -> np.ndarray:\n    boxes = boxes.astype(float, copy=True)\n    boxes[:, [0, 2]] = np.clip(boxes[:, [0, 2]], 0, max(width - 1, 0))\n    boxes[:, [1, 3]] = np.clip(boxes[:, [1, 3]], 0, max(height - 1, 0))\n    return boxes\n\n\ndef valid_box_mask(boxes: np.ndarray, min_size: float = 1.0) -> np.ndarray:\n    boxes = np.asarray(boxes, dtype=float)\n    if boxes.size == 0:\n        return np.zeros((0,), dtype=bool)\n    finite = np.isfinite(boxes).all(axis=1)\n    positive = (boxes[:, 2] - boxes[:, 0] >= min_size) & (boxes[:, 3] - boxes[:, 1] >= min_size)\n    non_negative = (boxes[:, 0] >= 0) & (boxes[:, 1] >= 0)\n    return finite & positive & non_negative\n\n\ndef box_iou_matrix(boxes1: np.ndarray, boxes2: np.ndarray) -> np.ndarray:\n    boxes1 = np.asarray(boxes1, dtype=float)\n    boxes2 = np.asarray(boxes2, dtype=float)\n    if boxes1.size == 0 or boxes2.size == 0:\n        return np.zeros((len(boxes1), len(boxes2)), dtype=float)\n\n    x1 = np.maximum(boxes1[:, None, 0], boxes2[None, :, 0])\n    y1 = np.maximum(boxes1[:, None, 1], boxes2[None, :, 1])\n    x2 = np.minimum(boxes1[:, None, 2], boxes2[None, :, 2])\n    y2 = np.minimum(boxes1[:, None, 3], boxes2[None, :, 3])\n\n    inter = np.maximum(0, x2 - x1) * np.maximum(0, y2 - y1)\n    area1 = np.maximum(0, boxes1[:, 2] - boxes1[:, 0]) * np.maximum(0, boxes1[:, 3] - boxes1[:, 1])\n    area2 = np.maximum(0, boxes2[:, 2] - boxes2[:, 0]) * np.maximum(0, boxes2[:, 3] - boxes2[:, 1])\n    union = area1[:, None] + area2[None, :] - inter\n    return np.divide(inter, union, out=np.zeros_like(inter, dtype=float), where=union > 0)\n\n\ndef classwise_nms(boxes, scores, labels, iou_threshold: float):\n    \"\"\"Run class-wise NMS on torch tensors and return kept indices.\"\"\"\n    import torch\n\n    if boxes.numel() == 0:\n        return torch.empty((0,), dtype=torch.long, device=boxes.device)\n\n    try:\n        from torchvision.ops import nms\n    except Exception:\n        nms = None\n\n    kept = []\n    for class_id in labels.unique():\n        class_indices = torch.where(labels == class_id)[0]\n        class_boxes = boxes[class_indices]\n        class_scores = scores[class_indices]\n        if nms is not None:\n            class_keep = class_indices[nms(class_boxes, class_scores, iou_threshold)]\n        else:\n            class_keep = _numpy_nms(class_indices, class_boxes, class_scores, iou_threshold)\n        kept.append(class_keep)\n\n    if not kept:\n        return torch.empty((0,), dtype=torch.long, device=boxes.device)\n    keep = torch.cat(kept)\n    return keep[torch.argsort(scores[keep], descending=True)]\n\n\ndef _numpy_nms(indices, boxes, scores, iou_threshold: float):\n    import torch\n\n    boxes_np = boxes.detach().cpu().numpy()\n    scores_np = scores.detach().cpu().numpy()\n    order = scores_np.argsort()[::-1]\n    keep_local = []\n\n    while order.size > 0:\n        i = order[0]\n        keep_local.append(i)\n        if order.size == 1:\n            break\n        ious = box_iou_matrix(boxes_np[[i]], boxes_np[order[1:]])[0]\n        order = order[1:][ious <= iou_threshold]\n\n    return indices[torch.as_tensor(keep_local, dtype=torch.long, device=indices.device)]\n",
  "src/utils/checkpoints.py": "\"\"\"Checkpoint save/load helpers.\"\"\"\n\nfrom __future__ import annotations\n\nfrom pathlib import Path\nfrom typing import Any\n\nimport torch\n\n\ndef save_checkpoint(\n    path: str | Path,\n    model: torch.nn.Module,\n    optimizer: torch.optim.Optimizer | None,\n    scaler: Any,\n    epoch: int,\n    best_score: float,\n    config: dict,\n) -> None:\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    payload = {\n        \"model\": model.state_dict(),\n        \"epoch\": epoch,\n        \"best_score\": best_score,\n        \"config\": config,\n    }\n    if optimizer is not None:\n        payload[\"optimizer\"] = optimizer.state_dict()\n    if scaler is not None:\n        payload[\"scaler\"] = scaler.state_dict()\n    torch.save(payload, path)\n\n\ndef load_checkpoint(\n    path: str | Path,\n    model: torch.nn.Module,\n    optimizer: torch.optim.Optimizer | None = None,\n    scaler: Any = None,\n    map_location: str | torch.device = \"cpu\",\n) -> dict:\n    checkpoint = torch.load(path, map_location=map_location)\n    state_dict = checkpoint.get(\"model\", checkpoint)\n    model.load_state_dict(state_dict)\n    if optimizer is not None and \"optimizer\" in checkpoint:\n        optimizer.load_state_dict(checkpoint[\"optimizer\"])\n    if scaler is not None and \"scaler\" in checkpoint:\n        scaler.load_state_dict(checkpoint[\"scaler\"])\n    return checkpoint\n",
  "src/utils/config.py": "\"\"\"Config loading and runtime override helpers.\"\"\"\n\nfrom __future__ import annotations\n\nimport copy\nimport os\nfrom pathlib import Path\nfrom typing import Any\n\nimport yaml\n\n\ndef load_config(path: str | Path) -> dict[str, Any]:\n    path = Path(path)\n    with path.open(\"r\", encoding=\"utf-8\") as f:\n        config = yaml.safe_load(f)\n    if not isinstance(config, dict):\n        raise ValueError(f\"Config must be a YAML mapping: {path}\")\n    return config\n\n\ndef apply_runtime_overrides(\n    config: dict[str, Any],\n    data_root: str | None = None,\n    work_dir: str | None = None,\n    smoke_test: bool = False,\n) -> dict[str, Any]:\n    cfg = copy.deepcopy(config)\n\n    env_data_root = os.environ.get(\"LGCXR_DATA_ROOT\")\n    env_work_dir = os.environ.get(\"LGCXR_WORK_DIR\")\n\n    if env_data_root:\n        cfg[\"data_root\"] = env_data_root\n    if env_work_dir:\n        cfg[\"work_dir\"] = env_work_dir\n    if data_root:\n        cfg[\"data_root\"] = data_root\n    if work_dir:\n        cfg[\"work_dir\"] = work_dir\n    if smoke_test:\n        cfg[\"smoke_test\"] = True\n\n    cfg = normalize_work_dir_paths(cfg)\n    if cfg.get(\"smoke_test\"):\n        cfg = apply_smoke_defaults(cfg)\n    return cfg\n\n\ndef normalize_work_dir_paths(config: dict[str, Any]) -> dict[str, Any]:\n    cfg = copy.deepcopy(config)\n    work_dir = Path(str(cfg[\"work_dir\"]))\n\n    scanner = cfg.setdefault(\"scanner\", {})\n    scanner[\"best_ckpt\"] = str(work_dir / Path(scanner.get(\"best_ckpt\", \"lgcxr_scanner_fasterrcnn_best.pth\")).name)\n    scanner[\"last_ckpt\"] = str(work_dir / Path(scanner.get(\"last_ckpt\", \"lgcxr_scanner_fasterrcnn_last.pth\")).name)\n\n    global_classifier = cfg.setdefault(\"global_classifier\", {})\n    global_classifier[\"checkpoint\"] = str(\n        work_dir / Path(global_classifier.get(\"checkpoint\", \"lgcxr_global_classifier_best.pth\")).name\n    )\n\n    crop_classifier = cfg.setdefault(\"crop_classifier\", {})\n    crop_classifier[\"checkpoint\"] = str(\n        work_dir / Path(crop_classifier.get(\"checkpoint\", \"lgcxr_crop_classifier_best.pth\")).name\n    )\n\n    submission = cfg.setdefault(\"submission\", {})\n    submission[\"output_path\"] = str(work_dir / Path(submission.get(\"output_path\", \"submission.csv\")).name)\n    return cfg\n\n\ndef apply_smoke_defaults(config: dict[str, Any]) -> dict[str, Any]:\n    cfg = copy.deepcopy(config)\n    scanner = cfg.setdefault(\"scanner\", {})\n    scanner[\"epochs\"] = min(int(scanner.get(\"epochs\", 1)), 1)\n    scanner[\"num_workers\"] = 0\n    scanner[\"batch_size\"] = min(int(scanner.get(\"batch_size\", 1)), 2)\n    return cfg\n\n\ndef ensure_work_dir(config: dict[str, Any]) -> Path:\n    work_dir = Path(str(config[\"work_dir\"]))\n    work_dir.mkdir(parents=True, exist_ok=True)\n    return work_dir\n\n\ndef output_path(config: dict[str, Any], filename: str) -> Path:\n    return Path(str(config[\"work_dir\"])) / filename\n",
  "src/utils/logging.py": "\"\"\"Small logging helper used by command-line scripts.\"\"\"\n\nfrom __future__ import annotations\n\nimport logging\nimport sys\n\n\ndef configure_logger(name: str = \"lgcxr\", level: int = logging.INFO) -> logging.Logger:\n    logger = logging.getLogger(name)\n    logger.setLevel(level)\n    logger.propagate = False\n\n    if not logger.handlers:\n        handler = logging.StreamHandler(sys.stdout)\n        handler.setFormatter(logging.Formatter(\"%(asctime)s | %(levelname)s | %(message)s\"))\n        logger.addHandler(handler)\n\n    return logger\n",
  "src/utils/metrics_voc.py": "\"\"\"VOC-style detection metrics used as a validation proxy.\"\"\"\n\nfrom __future__ import annotations\n\nfrom collections import defaultdict\nfrom typing import Iterable\n\nimport numpy as np\n\nfrom .boxes import box_iou_matrix\n\n\ndef voc_ap(recalls: np.ndarray, precisions: np.ndarray) -> float:\n    if recalls.size == 0:\n        return 0.0\n    mrec = np.concatenate(([0.0], recalls, [1.0]))\n    mpre = np.concatenate(([0.0], precisions, [0.0]))\n    for i in range(mpre.size - 1, 0, -1):\n        mpre[i - 1] = max(mpre[i - 1], mpre[i])\n    changing = np.where(mrec[1:] != mrec[:-1])[0]\n    return float(np.sum((mrec[changing + 1] - mrec[changing]) * mpre[changing + 1]))\n\n\ndef voc_map(\n    ground_truth: Iterable[dict],\n    predictions: Iterable[dict],\n    class_ids: Iterable[int] = range(14),\n    iou_threshold: float = 0.4,\n) -> dict:\n    gt_by_class_image = defaultdict(lambda: defaultdict(list))\n    pred_by_class = defaultdict(list)\n\n    for row in ground_truth:\n        gt_by_class_image[int(row[\"class_id\"])][str(row[\"image_id\"])].append(row[\"box\"])\n\n    for row in predictions:\n        pred_by_class[int(row[\"class_id\"])].append(row)\n\n    per_class_ap = {}\n    for class_id in class_ids:\n        gt_for_class = gt_by_class_image[class_id]\n        npos = sum(len(v) for v in gt_for_class.values())\n        if npos == 0:\n            per_class_ap[class_id] = None\n            continue\n\n        detected = {image_id: np.zeros(len(boxes), dtype=bool) for image_id, boxes in gt_for_class.items()}\n        preds = sorted(pred_by_class[class_id], key=lambda x: float(x[\"score\"]), reverse=True)\n        tp = np.zeros(len(preds), dtype=float)\n        fp = np.zeros(len(preds), dtype=float)\n\n        for i, pred in enumerate(preds):\n            image_id = str(pred[\"image_id\"])\n            gt_boxes = np.asarray(gt_for_class.get(image_id, []), dtype=float)\n            if gt_boxes.size == 0:\n                fp[i] = 1.0\n                continue\n\n            ious = box_iou_matrix(np.asarray([pred[\"box\"]], dtype=float), gt_boxes)[0]\n            best_idx = int(np.argmax(ious))\n            best_iou = float(ious[best_idx])\n            if best_iou >= iou_threshold and not detected[image_id][best_idx]:\n                tp[i] = 1.0\n                detected[image_id][best_idx] = True\n            else:\n                fp[i] = 1.0\n\n        fp_cum = np.cumsum(fp)\n        tp_cum = np.cumsum(tp)\n        recalls = tp_cum / max(float(npos), 1.0)\n        precisions = tp_cum / np.maximum(tp_cum + fp_cum, 1e-12)\n        per_class_ap[class_id] = voc_ap(recalls, precisions)\n\n    valid_aps = [ap for ap in per_class_ap.values() if ap is not None]\n    return {\n        \"map\": float(np.mean(valid_aps)) if valid_aps else 0.0,\n        \"per_class_ap\": per_class_ap,\n        \"iou_threshold\": iou_threshold,\n    }\n",
  "src/utils/seed.py": "\"\"\"Reproducibility helpers.\"\"\"\n\nfrom __future__ import annotations\n\nimport os\nimport random\n\nimport numpy as np\nimport torch\n\n\ndef set_seed(seed: int) -> None:\n    random.seed(seed)\n    np.random.seed(seed)\n    os.environ[\"PYTHONHASHSEED\"] = str(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.benchmark = True\n",
  "src/utils/visualization.py": "\"\"\"Debug visualization helpers.\"\"\"\n\nfrom __future__ import annotations\n\nfrom pathlib import Path\nfrom typing import Iterable\n\nfrom PIL import Image, ImageDraw, ImageFont\n\n\ndef draw_debug_annotation(\n    image_path: str | Path,\n    boxes: Iterable[Iterable[float]],\n    labels: Iterable[int],\n    output_path: str | Path,\n) -> Path:\n    image = Image.open(image_path).convert(\"RGB\")\n    draw = ImageDraw.Draw(image)\n    try:\n        font = ImageFont.load_default()\n    except Exception:\n        font = None\n\n    for box, label in zip(boxes, labels):\n        x1, y1, x2, y2 = [float(v) for v in box]\n        draw.rectangle([x1, y1, x2, y2], outline=\"red\", width=3)\n        draw.text((x1, max(0, y1 - 12)), str(label), fill=\"red\", font=font)\n\n    output_path = Path(output_path)\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    image.save(output_path)\n    return output_path\n",
  "scripts/00_preflight.py": "#!/usr/bin/env python\n\"\"\"Run dataset and submission preflight checks.\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport sys\nfrom pathlib import Path\n\nPROJECT_ROOT = Path(__file__).resolve().parents[1]\nsys.path.insert(0, str(PROJECT_ROOT))\n\nfrom src.data.preflight import run_preflight\nfrom src.utils.config import apply_runtime_overrides, load_config\nfrom src.utils.logging import configure_logger\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(description=\"Run LG-CXR Faster R-CNN preflight checks.\")\n    parser.add_argument(\"--config\", required=True, help=\"Path to YAML config.\")\n    parser.add_argument(\"--data-root\", default=None, help=\"Override dataset root.\")\n    parser.add_argument(\"--work-dir\", default=None, help=\"Override output directory.\")\n    parser.add_argument(\"--smoke-test\", action=\"store_true\", help=\"Apply smoke-test config overrides.\")\n    return parser.parse_args()\n\n\ndef main() -> int:\n    args = parse_args()\n    logger = configure_logger()\n    config = apply_runtime_overrides(\n        load_config(args.config),\n        data_root=args.data_root,\n        work_dir=args.work_dir,\n        smoke_test=args.smoke_test,\n    )\n    result = run_preflight(config)\n    logger.info(\"Preflight passed with %d checks.\", len(result[\"checks\"]))\n    return 0\n\n\nif __name__ == \"__main__\":\n    raise SystemExit(main())\n",
  "scripts/01_train_scanner.py": "#!/usr/bin/env python\n\"\"\"Train the Faster R-CNN scanner.\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport sys\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nimport pandas as pd\nimport torch\nfrom torch.utils.data import DataLoader\n\nPROJECT_ROOT = Path(__file__).resolve().parents[1]\nsys.path.insert(0, str(PROJECT_ROOT))\n\nfrom src.data.columns import infer_detection_columns\nfrom src.data.dataset import CXRDetectionDataset\nfrom src.data.discovery import discover_dataset, load_csv\nfrom src.data.image_paths import build_image_index, image_ids_from_paths\nfrom src.data.image_sizes import load_image_size_map\nfrom src.data.preflight import run_preflight\nfrom src.data.splits import make_train_val_split\nfrom src.models.fasterrcnn import build_fasterrcnn\nfrom src.train.train_scanner import detection_collate, evaluate_detector_map, make_optimizer, train_one_epoch\nfrom src.utils.checkpoints import load_checkpoint, save_checkpoint\nfrom src.utils.config import apply_runtime_overrides, ensure_work_dir, load_config, output_path\nfrom src.utils.logging import configure_logger\nfrom src.utils.seed import set_seed\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(description=\"Train LG-CXR Faster R-CNN scanner.\")\n    parser.add_argument(\"--config\", required=True, help=\"Path to YAML config.\")\n    parser.add_argument(\"--data-root\", default=None, help=\"Override dataset root.\")\n    parser.add_argument(\"--work-dir\", default=None, help=\"Override output directory.\")\n    parser.add_argument(\"--smoke-test\", action=\"store_true\", help=\"Train on small subsets for one epoch.\")\n    parser.add_argument(\"--skip-preflight\", action=\"store_true\", help=\"Skip preflight because it already passed.\")\n    return parser.parse_args()\n\n\ndef main() -> int:\n    args = parse_args()\n    logger = configure_logger()\n    config = apply_runtime_overrides(\n        load_config(args.config),\n        data_root=args.data_root,\n        work_dir=args.work_dir,\n        smoke_test=args.smoke_test,\n    )\n    work_dir = ensure_work_dir(config)\n    set_seed(int(config.get(\"seed\", 42)))\n\n    if not args.skip_preflight:\n        logger.info(\"Running preflight before training.\")\n        run_preflight(config)\n\n    discovery = discover_dataset(config[\"data_root\"])\n    if discovery.train_csv is None:\n        raise RuntimeError(\"No train CSV found after preflight.\")\n    train_df = load_csv(discovery.train_csv)\n    columns = infer_detection_columns(train_df)\n    original_size_map, size_columns = load_image_size_map(discovery.img_size_csv)\n    if original_size_map:\n        logger.info(\"Loaded original image sizes from %s using columns %s\", discovery.img_size_csv, size_columns)\n    else:\n        logger.warning(\"No img_size.csv mapping loaded; training boxes will be used without original-to-PNG scaling.\")\n    train_image_paths = discovery.train_image_paths or discovery.image_paths\n    image_index = build_image_index(train_image_paths)\n    all_train_image_ids = image_ids_from_paths(train_image_paths) if discovery.train_image_paths else None\n\n    train_ids, val_ids = make_train_val_split(\n        train_df,\n        columns,\n        val_size=float(config.get(\"val_size\", 0.2)),\n        seed=int(config.get(\"seed\", 42)),\n        all_image_ids=all_train_image_ids,\n        smoke_test=bool(config.get(\"smoke_test\", False)),\n        smoke_max_train_images=int(config.get(\"smoke_max_train_images\", 500)),\n        smoke_max_val_images=int(config.get(\"smoke_max_val_images\", 100)),\n    )\n    logger.info(\"Train images: %d | Val images: %d\", len(train_ids), len(val_ids))\n\n    no_finding_class_id = int(config[\"submission\"][\"no_finding_class_id\"])\n    train_dataset = CXRDetectionDataset(\n        train_df,\n        train_ids,\n        image_index,\n        columns,\n        no_finding_class_id,\n        original_size_map=original_size_map,\n    )\n    val_dataset = CXRDetectionDataset(\n        train_df,\n        val_ids,\n        image_index,\n        columns,\n        no_finding_class_id,\n        original_size_map=original_size_map,\n    )\n\n    scanner_cfg = config[\"scanner\"]\n    train_loader = DataLoader(\n        train_dataset,\n        batch_size=int(scanner_cfg[\"batch_size\"]),\n        shuffle=True,\n        num_workers=int(scanner_cfg[\"num_workers\"]),\n        collate_fn=detection_collate,\n    )\n    val_loader = DataLoader(\n        val_dataset,\n        batch_size=1,\n        shuffle=False,\n        num_workers=int(scanner_cfg[\"num_workers\"]),\n        collate_fn=detection_collate,\n    )\n\n    device = torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")\n    logger.info(\"Using device: %s\", device)\n    model = build_fasterrcnn(\n        num_classes=15,\n        image_size=int(scanner_cfg[\"image_size\"]),\n        max_size=int(scanner_cfg.get(\"max_size\", round(int(scanner_cfg[\"image_size\"]) * 1.5))),\n    )\n    model.to(device)\n\n    optimizer = make_optimizer(model, lr=float(scanner_cfg[\"lr\"]), weight_decay=float(scanner_cfg[\"weight_decay\"]))\n    amp_enabled = bool(scanner_cfg.get(\"amp\", True) and device.type == \"cuda\")\n    scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)\n\n    start_epoch = 0\n    best_score = -1.0\n    last_ckpt = Path(scanner_cfg[\"last_ckpt\"])\n    if bool(scanner_cfg.get(\"resume\", True)) and last_ckpt.exists():\n        logger.info(\"Resuming from last checkpoint: %s\", last_ckpt)\n        checkpoint = load_checkpoint(last_ckpt, model, optimizer=optimizer, scaler=scaler, map_location=device)\n        start_epoch = int(checkpoint.get(\"epoch\", -1)) + 1\n        best_score = float(checkpoint.get(\"best_score\", -1.0))\n\n    history = []\n    epochs = int(scanner_cfg[\"epochs\"])\n    for epoch in range(start_epoch, epochs):\n        train_loss = train_one_epoch(\n            model,\n            train_loader,\n            optimizer,\n            device,\n            amp=amp_enabled,\n            grad_clip=scanner_cfg.get(\"grad_clip\"),\n            scaler=scaler,\n            epoch=epoch,\n        )\n        if len(val_dataset) > 0:\n            metrics = evaluate_detector_map(\n                model,\n                val_loader,\n                val_dataset,\n                device,\n                iou_threshold=float(config[\"evaluation\"][\"voc_iou_threshold\"]),\n                conf_threshold=float(config[\"inference\"][\"conf_threshold\"]),\n            )\n            score = float(metrics[\"map\"])\n        else:\n            metrics = {\"map\": 0.0, \"per_class_ap\": {}, \"iou_threshold\": config[\"evaluation\"][\"voc_iou_threshold\"]}\n            score = -train_loss\n\n        save_checkpoint(scanner_cfg[\"last_ckpt\"], model, optimizer, scaler, epoch, best_score, config)\n        if score > best_score:\n            best_score = score\n            save_checkpoint(scanner_cfg[\"best_ckpt\"], model, optimizer, scaler, epoch, best_score, config)\n\n        row = {\"epoch\": epoch, \"train_loss\": train_loss, \"val_map_proxy\": score, \"metrics\": metrics}\n        history.append(row)\n        logger.info(\"Epoch %d | loss %.5f | val proxy %.5f | best %.5f\", epoch, train_loss, score, best_score)\n\n    if not Path(scanner_cfg[\"best_ckpt\"]).exists():\n        logger.info(\"Best checkpoint was not present; saving current model as best.\")\n        save_checkpoint(scanner_cfg[\"best_ckpt\"], model, optimizer, scaler, max(start_epoch - 1, 0), best_score, config)\n\n    summary = {\n        \"created_at\": datetime.now(timezone.utc).isoformat(),\n        \"train_csv\": str(discovery.train_csv),\n        \"train_images\": len(train_ids),\n        \"val_images\": len(val_ids),\n        \"device\": str(device),\n        \"best_score\": best_score,\n        \"history\": history,\n        \"config\": config,\n    }\n    summary_path = output_path(config, \"lgcxr_scanner_training_summary.json\")\n    summary_path.write_text(json.dumps(summary, indent=2), encoding=\"utf-8\")\n    logger.info(\"Training summary saved to %s\", summary_path)\n    logger.info(\"Best checkpoint: %s\", scanner_cfg[\"best_ckpt\"])\n    logger.info(\"Last checkpoint: %s\", scanner_cfg[\"last_ckpt\"])\n    return 0\n\n\nif __name__ == \"__main__\":\n    raise SystemExit(main())\n",
  "scripts/02_predict_scanner.py": "#!/usr/bin/env python\n\"\"\"Predict validation and test detections with the Faster R-CNN scanner.\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport sys\nfrom pathlib import Path\n\nimport pandas as pd\nimport torch\nfrom torch.utils.data import DataLoader\n\nPROJECT_ROOT = Path(__file__).resolve().parents[1]\nsys.path.insert(0, str(PROJECT_ROOT))\n\nfrom src.data.columns import infer_detection_columns, infer_image_id_column\nfrom src.data.dataset import CXRImageDataset\nfrom src.data.discovery import discover_dataset, load_csv\nfrom src.data.image_paths import build_image_index, image_ids_from_paths\nfrom src.data.image_sizes import load_image_size_map\nfrom src.data.splits import make_train_val_split, sample_submission_image_ids\nfrom src.inference.predict_scanner import image_collate, predict_dataset, save_predictions\nfrom src.models.fasterrcnn import build_fasterrcnn\nfrom src.utils.checkpoints import load_checkpoint\nfrom src.utils.config import apply_runtime_overrides, ensure_work_dir, load_config, output_path\nfrom src.utils.logging import configure_logger\nfrom src.utils.seed import set_seed\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(description=\"Predict LG-CXR Faster R-CNN detections.\")\n    parser.add_argument(\"--config\", required=True, help=\"Path to YAML config.\")\n    parser.add_argument(\"--data-root\", default=None, help=\"Override dataset root.\")\n    parser.add_argument(\"--work-dir\", default=None, help=\"Override output directory.\")\n    parser.add_argument(\"--smoke-test\", action=\"store_true\", help=\"Predict on small subsets.\")\n    return parser.parse_args()\n\n\ndef main() -> int:\n    args = parse_args()\n    logger = configure_logger()\n    config = apply_runtime_overrides(\n        load_config(args.config),\n        data_root=args.data_root,\n        work_dir=args.work_dir,\n        smoke_test=args.smoke_test,\n    )\n    ensure_work_dir(config)\n    set_seed(int(config.get(\"seed\", 42)))\n\n    scanner_cfg = config[\"scanner\"]\n    ckpt_path = Path(scanner_cfg[\"best_ckpt\"])\n    if not ckpt_path.exists():\n        raise FileNotFoundError(f\"Best scanner checkpoint not found: {ckpt_path}\")\n\n    discovery = discover_dataset(config[\"data_root\"])\n    if discovery.train_csv is None or discovery.sample_submission is None:\n        raise RuntimeError(\"Train CSV and sample submission are required for scanner prediction.\")\n\n    train_df = load_csv(discovery.train_csv)\n    sample_df = load_csv(discovery.sample_submission)\n    columns = infer_detection_columns(train_df)\n    original_size_map, size_columns = load_image_size_map(discovery.img_size_csv)\n    if original_size_map:\n        logger.info(\"Loaded original image sizes from %s using columns %s\", discovery.img_size_csv, size_columns)\n    else:\n        logger.warning(\"No img_size.csv mapping loaded; predictions will stay in PNG/model image space.\")\n    train_image_paths = discovery.train_image_paths or discovery.image_paths\n    test_image_paths = discovery.test_image_paths or discovery.image_paths\n    train_image_index = build_image_index(train_image_paths)\n    test_image_index = build_image_index(test_image_paths)\n    all_train_image_ids = image_ids_from_paths(train_image_paths) if discovery.train_image_paths else None\n\n    _, val_ids = make_train_val_split(\n        train_df,\n        columns,\n        val_size=float(config.get(\"val_size\", 0.2)),\n        seed=int(config.get(\"seed\", 42)),\n        all_image_ids=all_train_image_ids,\n        smoke_test=bool(config.get(\"smoke_test\", False)),\n        smoke_max_train_images=int(config.get(\"smoke_max_train_images\", 500)),\n        smoke_max_val_images=int(config.get(\"smoke_max_val_images\", 100)),\n    )\n    test_ids = _test_ids_from_metadata_or_sample(discovery, sample_df, config)\n\n    device = torch.device(\"cuda\" if torch.cuda.is_available() else \"cpu\")\n    model = build_fasterrcnn(\n        num_classes=15,\n        image_size=int(scanner_cfg[\"image_size\"]),\n        max_size=int(scanner_cfg.get(\"max_size\", round(int(scanner_cfg[\"image_size\"]) * 1.5))),\n    )\n    load_checkpoint(ckpt_path, model, map_location=device)\n    model.to(device)\n\n    inference_cfg = config[\"inference\"]\n    val_df = _predict_ids(\n        model,\n        val_ids,\n        train_image_index,\n        device,\n        inference_cfg,\n        int(scanner_cfg[\"num_workers\"]),\n        original_size_map=original_size_map,\n    )\n    test_df = _predict_ids(\n        model,\n        test_ids,\n        test_image_index,\n        device,\n        inference_cfg,\n        int(scanner_cfg[\"num_workers\"]),\n        original_size_map=original_size_map,\n    )\n\n    val_path = save_predictions(val_df, output_path(config, \"lgcxr_fast_val_predictions.csv\"))\n    test_path = save_predictions(test_df, output_path(config, \"lgcxr_fast_test_predictions.csv\"))\n    logger.info(\"Validation predictions saved to %s (%d rows)\", val_path, len(val_df))\n    logger.info(\"Test predictions saved to %s (%d rows)\", test_path, len(test_df))\n    return 0\n\n\ndef _predict_ids(model, image_ids, image_index, device, inference_cfg, num_workers: int, original_size_map=None) -> pd.DataFrame:\n    if not image_ids:\n        return pd.DataFrame(columns=[\"image_id\", \"class_id\", \"confidence\", \"xmin\", \"ymin\", \"xmax\", \"ymax\"])\n    dataset = CXRImageDataset(image_ids, image_index)\n    loader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=num_workers, collate_fn=image_collate)\n    return predict_dataset(\n        model,\n        loader,\n        device,\n        conf_threshold=float(inference_cfg[\"conf_threshold\"]),\n        nms_iou=float(inference_cfg[\"nms_iou\"]),\n        max_det_per_image=int(inference_cfg[\"max_det_per_image\"]),\n        original_size_map=original_size_map,\n    )\n\n\ndef _test_ids_from_metadata_or_sample(discovery, sample_df, config) -> list[str]:\n    smoke_test = bool(config.get(\"smoke_test\", False))\n    smoke_max_test_images = int(config.get(\"smoke_max_test_images\", 100))\n    if discovery.test_csv is not None:\n        test_df = load_csv(discovery.test_csv)\n        try:\n            image_col = infer_image_id_column(test_df)\n            test_ids = [str(v) for v in test_df[image_col].dropna().astype(str).tolist()]\n            if smoke_test:\n                test_ids = test_ids[:smoke_max_test_images]\n            if test_ids:\n                return test_ids\n        except Exception:\n            pass\n    return sample_submission_image_ids(sample_df, smoke_test=smoke_test, smoke_max_test_images=smoke_max_test_images)\n\n\nif __name__ == \"__main__\":\n    raise SystemExit(main())\n",
  "scripts/03_make_submission.py": "#!/usr/bin/env python\n\"\"\"Create Kaggle submission.csv from test predictions.\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport sys\nfrom pathlib import Path\n\nimport pandas as pd\n\nPROJECT_ROOT = Path(__file__).resolve().parents[1]\nsys.path.insert(0, str(PROJECT_ROOT))\n\nfrom src.data.discovery import discover_dataset, load_csv\nfrom src.inference.make_submission import make_submission\nfrom src.utils.config import apply_runtime_overrides, ensure_work_dir, load_config, output_path\nfrom src.utils.logging import configure_logger\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(description=\"Create LG-CXR submission.csv.\")\n    parser.add_argument(\"--config\", required=True, help=\"Path to YAML config.\")\n    parser.add_argument(\"--data-root\", default=None, help=\"Override dataset root.\")\n    parser.add_argument(\"--work-dir\", default=None, help=\"Override output directory.\")\n    return parser.parse_args()\n\n\ndef main() -> int:\n    args = parse_args()\n    logger = configure_logger()\n    config = apply_runtime_overrides(load_config(args.config), data_root=args.data_root, work_dir=args.work_dir)\n    ensure_work_dir(config)\n\n    discovery = discover_dataset(config[\"data_root\"])\n    if discovery.sample_submission is None:\n        raise RuntimeError(\"Could not identify sample submission CSV.\")\n\n    sample_df = load_csv(discovery.sample_submission)\n    prediction_path = output_path(config, \"lgcxr_fast_test_predictions.csv\")\n    if not prediction_path.exists():\n        raise FileNotFoundError(f\"Test prediction CSV not found: {prediction_path}\")\n    predictions = pd.read_csv(prediction_path)\n\n    output = config[\"submission\"][\"output_path\"]\n    submission = make_submission(\n        sample_df,\n        predictions,\n        output,\n        no_finding_string=str(config[\"submission\"][\"no_finding_string\"]),\n    )\n    logger.info(\"Submission saved to %s (%d rows)\", output, len(submission))\n    return 0\n\n\nif __name__ == \"__main__\":\n    raise SystemExit(main())\n",
  "scripts/04_full_pipeline.py": "#!/usr/bin/env python\n\"\"\"Run preflight, training, prediction, and submission creation.\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport subprocess\nimport sys\nfrom pathlib import Path\n\nPROJECT_ROOT = Path(__file__).resolve().parents[1]\nsys.path.insert(0, str(PROJECT_ROOT))\n\nfrom src.utils.config import apply_runtime_overrides, load_config\nfrom src.utils.logging import configure_logger\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(description=\"Run the full LG-CXR Faster R-CNN pipeline.\")\n    parser.add_argument(\"--config\", required=True, help=\"Path to YAML config.\")\n    parser.add_argument(\"--data-root\", default=None, help=\"Override dataset root.\")\n    parser.add_argument(\"--work-dir\", default=None, help=\"Override output directory.\")\n    parser.add_argument(\"--smoke-test\", action=\"store_true\", help=\"Run small one-epoch smoke pipeline.\")\n    parser.add_argument(\"--force-train\", action=\"store_true\", help=\"Train even if best checkpoint already exists.\")\n    return parser.parse_args()\n\n\ndef main() -> int:\n    args = parse_args()\n    logger = configure_logger()\n    config_path = str(Path(args.config))\n    config = apply_runtime_overrides(\n        load_config(config_path),\n        data_root=args.data_root,\n        work_dir=args.work_dir,\n        smoke_test=args.smoke_test,\n    )\n\n    common = [\"--config\", config_path]\n    if args.data_root:\n        common += [\"--data-root\", args.data_root]\n    if args.work_dir:\n        common += [\"--work-dir\", args.work_dir]\n    if args.smoke_test:\n        common += [\"--smoke-test\"]\n\n    _run(\"preflight\", [sys.executable, \"scripts/00_preflight.py\", *common])\n\n    best_ckpt = Path(config[\"scanner\"][\"best_ckpt\"])\n    if args.force_train or not best_ckpt.exists():\n        logger.info(\"Training scanner because best checkpoint is missing or force-train was requested.\")\n        _run(\"train scanner\", [sys.executable, \"scripts/01_train_scanner.py\", *common, \"--skip-preflight\"])\n    else:\n        logger.info(\"Best checkpoint exists; skipping training: %s\", best_ckpt)\n\n    _run(\"predict scanner\", [sys.executable, \"scripts/02_predict_scanner.py\", *common])\n    make_submission_common = [item for item in common if item != \"--smoke-test\"]\n    _run(\"make submission\", [sys.executable, \"scripts/03_make_submission.py\", *make_submission_common])\n    logger.info(\"Full pipeline complete.\")\n    return 0\n\n\ndef _run(name: str, command: list[str]) -> None:\n    print(f\"\\n=== {name} ===\")\n    subprocess.run(command, cwd=PROJECT_ROOT, check=True)\n\n\nif __name__ == \"__main__\":\n    raise SystemExit(main())\n",
  "scripts/05_audit_dimensions.py": "#!/usr/bin/env python\n\"\"\"Audit image dimensions and train box scale for resize decisions.\"\"\"\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport sys\nfrom pathlib import Path\n\nPROJECT_ROOT = Path(__file__).resolve().parents[1]\nsys.path.insert(0, str(PROJECT_ROOT))\n\nfrom src.data.dimension_audit import run_dimension_audit\nfrom src.utils.config import apply_runtime_overrides, load_config\nfrom src.utils.logging import configure_logger\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(description=\"Audit LG-CXR image dimensions and box scale.\")\n    parser.add_argument(\"--config\", required=True, help=\"Path to YAML config.\")\n    parser.add_argument(\"--data-root\", default=None, help=\"Override dataset root.\")\n    parser.add_argument(\"--work-dir\", default=None, help=\"Override output directory.\")\n    parser.add_argument(\n        \"--max-images\",\n        type=int,\n        default=0,\n        help=\"Maximum train/test images to open for PIL dimension checks. Use 0 for all images.\",\n    )\n    return parser.parse_args()\n\n\ndef main() -> int:\n    args = parse_args()\n    logger = configure_logger()\n    config = apply_runtime_overrides(load_config(args.config), data_root=args.data_root, work_dir=args.work_dir)\n    report = run_dimension_audit(config, max_images=int(args.max_images))\n    logger.info(\"Dimension audit saved to %s\", report[\"output_path\"])\n    logger.info(\"Current resize: %s\", json.dumps(report[\"recommendation\"][\"current\"]))\n    logger.info(\"Suggested next experiment: %s\", json.dumps(report[\"recommendation\"][\"suggested_next_experiment\"]))\n    for note in report[\"recommendation\"][\"notes\"]:\n        logger.info(\"Recommendation note: %s\", note)\n    return 0\n\n\nif __name__ == \"__main__\":\n    raise SystemExit(main())\n"
}

for relative_path, content in PROJECT_FILES.items():
    path = PROJECT_DIR / relative_path
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding='utf-8')

os.chdir(PROJECT_DIR)
print('Created project at', PROJECT_DIR)
print('Files written:', len(PROJECT_FILES))

## 3. Install Dependencies

In [ ]:
!pip install -r requirements.txt

## 4. Check Environment

In [ ]:
import platform
import torch
import torchvision

print('Python:', platform.python_version())
print('Torch:', torch.__version__)
print('Torchvision:', torchvision.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    x = torch.ones((2, 2), device='cuda')
    print('CUDA tensor test:', x.sum().item())

## 5. Configure Dataset and Output Paths

Default assumes the dataset has been placed in Drive. Edit `DATA_ROOT` if your folder is somewhere else. Outputs are saved to Drive so they survive runtime resets.

In [ ]:
import os
from pathlib import Path

PROJECT_DIR = '/content/amia-lgcxr-frcnn'
DATA_ROOT = '/content/drive/MyDrive/datasets/amia-public-challenge-2026'
WORK_DIR = '/content/drive/MyDrive/amia-lgcxr-frcnn/outputs'

# Kaggle-style alternatives:
# DATA_ROOT = '/kaggle/input/amia-public-challenge-2026'
# WORK_DIR = '/kaggle/working'

os.environ['LGCXR_DATA_ROOT'] = DATA_ROOT
os.environ['LGCXR_WORK_DIR'] = WORK_DIR
Path(WORK_DIR).mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_DIR)

print('PROJECT_DIR =', PROJECT_DIR)
print('DATA_ROOT =', DATA_ROOT)
print('WORK_DIR =', WORK_DIR)

## Optional: Download Dataset with Kaggle API

Run this only if you have Kaggle API credentials available. Competition access may require accepting the rules on Kaggle first.

In [ ]:
# Optional Kaggle API setup. Edit paths as needed, then uncomment.
# !pip install kaggle
# !mkdir -p ~/.kaggle
# !cp /content/drive/MyDrive/kaggle.json ~/.kaggle/kaggle.json
# !chmod 600 ~/.kaggle/kaggle.json
# !mkdir -p /content/data
# !kaggle competitions download -c amia-public-challenge-2026 -p /content/data
# !unzip -q /content/data/amia-public-challenge-2026.zip -d /content/data/amia-public-challenge-2026
# DATA_ROOT = '/content/data/amia-public-challenge-2026'
# os.environ['LGCXR_DATA_ROOT'] = DATA_ROOT
# print('DATA_ROOT =', DATA_ROOT)

## 6. Run Preflight

In [ ]:
!python scripts/00_preflight.py --config configs/baseline_frcnn.yaml

## 7. Audit Dimensions

Run this before changing `scanner.image_size` or `scanner.max_size`. The audit also checks the original-coordinate issue: `train.csv` boxes are in original scan space, while the PNG files may be resized.

In [ ]:
!python scripts/05_audit_dimensions.py --config configs/baseline_frcnn.yaml

## 8. Smoke Test

This uses small subsets, trains for at most one epoch, and creates a format-valid `submission.csv`.

In [ ]:
!python scripts/04_full_pipeline.py --config configs/baseline_frcnn.yaml --smoke-test

## 9. Full Training

In [ ]:
!python scripts/01_train_scanner.py --config configs/baseline_frcnn.yaml

## 10. Inference

In [ ]:
!python scripts/02_predict_scanner.py --config configs/baseline_frcnn.yaml

## 11. Make Submission

In [ ]:
!python scripts/03_make_submission.py --config configs/baseline_frcnn.yaml

## 12. Full Pipeline

In [ ]:
!python scripts/04_full_pipeline.py --config configs/baseline_frcnn.yaml

## 13. Inspect Outputs

In [ ]:
from pathlib import Path
import os

for path in sorted(Path(os.environ['LGCXR_WORK_DIR']).glob('*')):
    print(path)